<a href="https://colab.research.google.com/github/mhirschberg/competitive_vsibility_audit_bd/blob/main/ompetitive_visibility_audit_bd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title 1. Install dependencies
#@markdown Run this cell once after opening the notebook.

!pip install -q \
    requests \
    pydantic \
    pandas \
    rich \
    tldextract \
    json-repair \
    nest_asyncio

print("✓ Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.3 MB/s eta 0:00:00
✓ Dependencies installed


In [3]:
#@title 2. Configure the audit
#@markdown Enter the company you want to analyze.
#@markdown
#@markdown The domain can be entered as:
#@markdown - `example.com`
#@markdown - `www.example.com`
#@markdown - `https://www.example.com/`
#@markdown
#@markdown Your Bright Data API token is read from the Colab secret named `BRIGHTDATA_API_TOKEN`.

COMPANY_NAME = "Bright Data" #@param {type:"string"}

COMPANY_DOMAIN = "brightdata.com" #@param {type:"string"}

AUDIT_FOCUS = "" #@param {type:"string"}

COUNTRY = "US" #@param {type:"string"}

SERP_ZONE = "serp_api1" #@param {type:"string"}

AUTO_DOWNLOAD_REPORT = False #@param {type:"boolean"}

DEBUG_MODE = True #@param {type:"boolean"}


# ============================================================
# Validate configuration
# ============================================================

from urllib.parse import urlparse
from google.colab import userdata


def read_colab_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else ""
    except Exception:
        return ""


BRIGHTDATA_API_TOKEN = read_colab_secret(
    "BRIGHTDATA_API_TOKEN"
)

COMPANY_NAME = COMPANY_NAME.strip()
COMPANY_DOMAIN = COMPANY_DOMAIN.strip()
COUNTRY = COUNTRY.strip().upper()
SERP_ZONE = SERP_ZONE.strip()


if not BRIGHTDATA_API_TOKEN:
    raise ValueError(
        "The BRIGHTDATA_API_TOKEN Colab secret is missing. "
        "Add it through the key icon in the Colab sidebar and "
        "enable notebook access."
    )

if not COMPANY_NAME:
    raise ValueError(
        "COMPANY_NAME cannot be empty."
    )

if not COMPANY_DOMAIN:
    raise ValueError(
        "COMPANY_DOMAIN cannot be empty."
    )

if not COUNTRY:
    raise ValueError(
        "COUNTRY cannot be empty."
    )

if not SERP_ZONE:
    raise ValueError(
        "SERP_ZONE cannot be empty."
    )


# Accept either a domain or complete URL.
if not COMPANY_DOMAIN.lower().startswith(
    ("http://", "https://")
):
    COMPANY_URL = (
        f"https://{COMPANY_DOMAIN}"
    )
else:
    COMPANY_URL = COMPANY_DOMAIN


parsed_company_url = urlparse(
    COMPANY_URL
)

if not parsed_company_url.hostname:
    raise ValueError(
        f"Invalid company domain or URL: "
        f"{COMPANY_DOMAIN}"
    )


COMPANY_HOSTNAME = (
    parsed_company_url.hostname
    .lower()
    .removeprefix("www.")
)

COMPANY_URL = (
    f"{parsed_company_url.scheme or 'https'}://"
    f"{parsed_company_url.hostname}/"
)


# ============================================================
# Final user-facing configuration object
# ============================================================

AUDIT_SETTINGS = {
    "company_name": COMPANY_NAME,
    "company_url": COMPANY_URL,
    "company_domain": COMPANY_HOSTNAME,
    "audit_focus": AUDIT_FOCUS.strip(),
    "country": COUNTRY,
    "serp_zone": SERP_ZONE,
    "auto_download": (
        AUTO_DOWNLOAD_REPORT
    ),
    "debug": DEBUG_MODE,
}


print("✓ Audit configured")
print()
print(f"Company: {COMPANY_NAME}")
print(f"Website: {COMPANY_URL}")
print(f"Country: {COUNTRY}")
print(f"Audit focus: {AUDIT_FOCUS.strip() or 'Primary offering'}")
print(f"SERP zone: {SERP_ZONE}")
print(
    f"Debug logging: "
    f"{'enabled' if DEBUG_MODE else 'disabled'}"
)


✓ Audit configured

Company: Bright Data
Website: https://brightdata.com/
Country: US
Audit focus: Primary offering
SERP zone: serp_api1
Debug logging: enabled


In [4]:
#@title 3A. Load core audit engine
#@markdown Internal API, reliable snapshot handling, parsing, models, and SERP aggregation.
#@markdown This cell normally does not need to be edited.

import os
import re
import json
import time
import shutil
import asyncio
import zipfile

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import (
    quote_plus,
    urlparse,
    urlunparse,
)

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed,
)
from collections import (
    Counter,
    defaultdict,
)

import requests
import pandas as pd
import nest_asyncio
import tldextract

from json_repair import repair_json
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown

nest_asyncio.apply()
console = Console()


# ============================================================
# Constants
# ============================================================

GOOGLE_AI_MODE_DATASET_ID = (
    "gd_mcswdt6z2elth3zqr2"
)

CHATGPT_DATASET_ID = (
    "gd_m7aof0k82r803d5bjm"
)

GEMINI_DATASET_ID = (
    "gd_mbz66arm2mf9cu856y"
)

BD_REQUEST_URL = (
    "https://api.brightdata.com/request"
)

BD_SCRAPE_URL = (
    "https://api.brightdata.com/datasets/v3/scrape"
)

BD_TRIGGER_URL = (
    "https://api.brightdata.com/datasets/v3/trigger"
)

BD_PROGRESS_URL = (
    "https://api.brightdata.com/datasets/v3/progress"
)

BD_SNAPSHOT_URL = (
    "https://api.brightdata.com/datasets/v3/snapshot"
)

GOOGLE_AI_OUTPUT_FIELDS = (
    "prompt,"
    "answer_text,"
    "citations,"
    "answer_text_markdown,"
    "timestamp"
)

FAILED_STATUSES = {
    "failed",
    "error",
    "canceled",
    "cancelled",
    "aborted",
}


# ============================================================
# Exceptions
# ============================================================

class BrightDataAPIError(RuntimeError):
    pass


class SnapshotTimeoutError(TimeoutError):
    def __init__(
        self,
        snapshot_id,
        timeout_seconds,
    ):
        self.snapshot_id = snapshot_id
        self.timeout_seconds = (
            timeout_seconds
        )

        super().__init__(
            f"Snapshot {snapshot_id} did not finish "
            f"within {timeout_seconds} seconds."
        )


# ============================================================
# Data models
# ============================================================

class BuyerIntentKeyword(BaseModel):
    keyword: str
    intent: str = "commercial"
    rationale: str = ""


class BrandAnalysis(BaseModel):
    brand_name: str
    official_url: str
    domain: str
    category: str = ""
    description: str = ""
    positioning: str = ""

    target_customers: list[str] = Field(
        default_factory=list
    )

    products: list[str] = Field(
        default_factory=list
    )

    key_features: list[str] = Field(
        default_factory=list
    )

    differentiators: list[str] = Field(
        default_factory=list
    )

    confidence: float = 0.0

    evidence: list[str] = Field(
        default_factory=list
    )


class CompanyIntake(BaseModel):
    brand: BrandAnalysis

    buyer_intent_keywords: list[
        BuyerIntentKeyword
    ] = Field(default_factory=list)


class CompetitorCandidate(BaseModel):
    domain: str
    preferred_hostname: str
    homepage_url: str
    frequency: int
    keyword_coverage: float
    best_rank: int
    average_rank: float
    rank_score: float
    total_score: float

    matched_keywords: list[str] = Field(
        default_factory=list
    )

    serp_urls: list[str] = Field(
        default_factory=list
    )

    serp_titles: list[str] = Field(
        default_factory=list
    )


class SelectedCompetitor(BaseModel):
    brand_name: str
    domain: str
    official_url: str
    reason: str = ""
    confidence: float = 0.0


class BrandProfile(BaseModel):
    brand_name: str
    official_url: str
    domain: str
    category: str = ""
    positioning: str = ""

    target_customers: list[str] = Field(
        default_factory=list
    )

    relevant_products: list[str] = Field(
        default_factory=list
    )

    key_features: list[str] = Field(
        default_factory=list
    )

    differentiators: list[str] = Field(
        default_factory=list
    )

    pricing_model: str = "unknown"
    competitor_reason: str = ""
    direct_competitor: bool = True
    confidence: float = 0.0

    evidence: list[str] = Field(
        default_factory=list
    )


# ============================================================
# General helpers
# ============================================================

def model_to_dict(model):
    if hasattr(model, "model_dump"):
        return model.model_dump()

    return model.dict()


def validate_model(
    model_class,
    data,
):
    if hasattr(
        model_class,
        "model_validate",
    ):
        return model_class.model_validate(
            data
        )

    return model_class.parse_obj(data)


def ensure_string_list(value):
    if value is None:
        return []

    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []

    if isinstance(value, list):
        results = []

        for item in value:
            if item is None:
                continue

            item = str(item).strip()

            if item:
                results.append(item)

        return results

    return [str(value).strip()]


def normalize_boolean(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        return value.strip().lower() in {
            "true",
            "yes",
            "1",
        }

    return bool(value)


def normalize_confidence(value):
    try:
        value = float(value)

        if 1 < value <= 100:
            value = value / 100

        return max(
            0.0,
            min(1.0, value),
        )

    except (TypeError, ValueError):
        return 0.0


def shorten(
    value,
    max_length,
):
    value = str(value or "").strip()

    if len(value) <= max_length:
        return value

    return (
        value[:max_length - 3]
        .rstrip()
        + "..."
    )


def slugify(value):
    value = re.sub(
        r"[^a-zA-Z0-9]+",
        "-",
        str(value or "").lower(),
    )

    return value.strip("-") or "audit"


def remove_ai_boilerplate(text):
    """
    Remove common UI text accidentally captured from AI interfaces.
    """
    if not isinstance(text, str):
        return ""

    unwanted_lines = {
        "log in",
        "login",
        "sign up",
        "sign up for free",
        "log insign up for free",
        "log in for more personalized help",
    }

    cleaned_lines = []

    for line in text.splitlines():
        normalized = re.sub(
            r"\s+",
            " ",
            line,
        ).strip().lower()

        if not normalized:
            cleaned_lines.append(line)
            continue

        if normalized in unwanted_lines:
            continue

        if (
            normalized.startswith(
                "log in for more personalized"
            )
        ):
            continue

        cleaned_lines.append(line)

    return "\n".join(
        cleaned_lines
    ).strip()


# ============================================================
# URL and domain helpers
# ============================================================

def extract_visible_url(value):
    value = str(value or "").strip()

    markdown_match = re.search(
        r"\[(https?://[^\]]+)\]"
        r"\([^)]+\)",
        value,
    )

    if markdown_match:
        value = markdown_match.group(1)

    url_match = re.search(
        r"https?://[^\s\])\"'>]+",
        value,
    )

    if url_match:
        value = url_match.group(0)

    return value.rstrip(
        ".,;:!?)]}"
    )


def normalize_public_url(value):
    value = extract_visible_url(value)

    if not value:
        value = str(value or "").strip()

    if not value:
        return ""

    if not value.lower().startswith(
        ("http://", "https://")
    ):
        value = f"https://{value}"

    parsed = urlparse(value)

    if not parsed.hostname:
        return ""

    scheme = parsed.scheme or "https"
    hostname = parsed.hostname.lower()
    path = parsed.path or "/"

    return f"{scheme}://{hostname}{path}"


def get_hostname(value):
    normalized = normalize_public_url(
        value
    )

    if not normalized:
        return ""

    return (
        urlparse(normalized).hostname
        or ""
    ).lower().removeprefix("www.")


def get_root_domain(value):
    if not value:
        return ""

    value = str(value).strip().lower()

    if "://" in value:
        hostname = (
            urlparse(value).hostname
            or ""
        )
    else:
        hostname = value.split("/")[0]

    hostname = hostname.removeprefix(
        "www."
    )

    extracted = tldextract.extract(
        hostname
    )

    if not extracted.domain:
        return hostname

    if not extracted.suffix:
        return extracted.domain

    return (
        f"{extracted.domain}."
        f"{extracted.suffix}"
    )


def canonical_source_url(value):
    """
    Deduplicate citation URLs by removing query strings and fragments.
    """
    value = str(value or "").strip()

    if not value:
        return ""

    try:
        parsed = urlparse(value)

        return urlunparse(
            (
                parsed.scheme,
                parsed.netloc.lower(),
                parsed.path.rstrip("/"),
                "",
                "",
                "",
            )
        )

    except Exception:
        return value


# ============================================================
# JSON repair
# ============================================================

def clean_ai_json_text(text):
    text = str(text or "").strip()

    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```\s*$",
        "",
        text,
    )

    replacements = {
        r"\_": "_",
        r"\[": "[",
        r"\]": "]",
        r"\{": "{",
        r"\}": "}",
        r"\#": "#",
        r"\-": "-",
        r"\+": "+",
        r"\.": ".",
    }

    for source, replacement in (
        replacements.items()
    ):
        text = text.replace(
            source,
            replacement,
        )

    text = re.sub(
        r"\[(https?://[^\]]+)\]"
        r"\([^)]+\)",
        r"\1",
        text,
    )

    return text.strip()


def parse_ai_json(text):
    cleaned = clean_ai_json_text(
        text
    )

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if (
        first_brace >= 0
        and last_brace > first_brace
    ):
        cleaned = cleaned[
            first_brace:last_brace + 1
        ]

    try:
        parsed = json.loads(cleaned)

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass

    repaired = repair_json(
        cleaned,
        return_objects=True,
    )

    if isinstance(repaired, dict):
        return repaired

    if isinstance(repaired, str):
        repaired = json.loads(repaired)

        if isinstance(repaired, dict):
            return repaired

    raise ValueError(
        "AI response could not be parsed "
        "as a JSON object."
    )


# ============================================================
# Bright Data client
# ============================================================

class BrightDataClient:
    def __init__(
        self,
        token,
        serp_zone,
        country="US",
        debug=False,
    ):
        self.token = token
        self.serp_zone = serp_zone
        self.country = (
            country or "US"
        ).upper()
        self.debug = bool(debug)

        self.headers = {
            "Authorization": (
                f"Bearer {self.token}"
            ),
            "Content-Type": (
                "application/json"
            ),
        }

    def log(
        self,
        message,
        style="dim",
    ):
        if self.debug:
            console.print(
                f"[{style}]{message}"
                f"[/{style}]"
            )

    @staticmethod
    def normalize_records(data):
        if data is None:
            return []

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            for key in (
                "data",
                "results",
                "records",
            ):
                if isinstance(
                    data.get(key),
                    list,
                ):
                    return data[key]

            return [data]

        return []

    @staticmethod
    def answer_text(record):
        if not isinstance(record, dict):
            return str(record or "").strip()

        value = (
            record.get(
                "answer_text_markdown"
            )
            or record.get(
                "answer_text"
            )
            or record.get(
                "answer_markdown"
            )
            or record.get("answer")
            or record.get("response")
            or record.get("text")
            or ""
        )

        if isinstance(value, str):
            return value.strip()

        if isinstance(
            value,
            (dict, list),
        ):
            return json.dumps(
                value,
                ensure_ascii=False,
            )

        return str(value or "").strip()

    def snapshot_status(
        self,
        snapshot_id,
    ):
        response = requests.get(
            f"{BD_PROGRESS_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            timeout=30,
        )

        if not response.ok:
            return {
                "status": "unknown",
                "details": (
                    response.text[:1000]
                ),
            }

        data = response.json()

        return {
            "status": str(
                data.get(
                    "status",
                    "unknown",
                )
            ).lower(),
            "details": data,
        }

    def download_snapshot(
        self,
        snapshot_id,
    ):
        response = requests.get(
            f"{BD_SNAPSHOT_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            params={"format": "json"},
            timeout=90,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"Could not download snapshot "
                f"{snapshot_id}. HTTP "
                f"{response.status_code}: "
                f"{response.text[:1500]}"
            )

        return self.normalize_records(
            response.json()
        )

    def wait_for_snapshot(
        self,
        snapshot_id,
        timeout_seconds=600,
        poll_seconds=5,
    ):
        started_at = time.monotonic()
        last_debug_log = -30

        while True:
            elapsed = (
                time.monotonic()
                - started_at
            )

            if elapsed >= timeout_seconds:
                raise SnapshotTimeoutError(
                    snapshot_id,
                    timeout_seconds,
                )

            status_result = (
                self.snapshot_status(
                    snapshot_id
                )
            )

            status = status_result[
                "status"
            ]

            if (
                self.debug
                and elapsed
                - last_debug_log
                >= 30
            ):
                self.log(
                    f"Snapshot {snapshot_id}: "
                    f"{status} — "
                    f"{elapsed:.0f}s"
                )

                last_debug_log = elapsed

            if status == "ready":
                records = (
                    self.download_snapshot(
                        snapshot_id
                    )
                )

                # A ready snapshot can briefly return
                # a materialization-status object.
                if (
                    len(records) == 1
                    and isinstance(
                        records[0],
                        dict,
                    )
                    and str(
                        records[0].get(
                            "status",
                            "",
                        )
                    ).lower()
                    in {
                        "building",
                        "collecting",
                        "digesting",
                        "running",
                    }
                ):
                    time.sleep(
                        poll_seconds
                    )
                    continue

                return records

            if status in FAILED_STATUSES:
                raise BrightDataAPIError(
                    f"Snapshot {snapshot_id} "
                    f"ended with status "
                    f"{status}."
                )

            time.sleep(poll_seconds)

    def scrape_dataset(
        self,
        dataset_id,
        payload,
        timeout_seconds=600,
        custom_output_fields=None,
    ):
        params = {
            "dataset_id": dataset_id,
            "format": "json",
            "notify": "false",
            "include_errors": "true",
        }

        if custom_output_fields:
            params[
                "custom_output_fields"
            ] = custom_output_fields

        response = requests.post(
            BD_SCRAPE_URL,
            headers=self.headers,
            params=params,
            json=payload,
            timeout=90,
        )

        if response.status_code not in {
            200,
            202,
        }:
            raise BrightDataAPIError(
                f"Dataset request failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:2000]}"
            )

        try:
            data = response.json()
        except Exception as exc:
            raise BrightDataAPIError(
                "Dataset response was not "
                "valid JSON."
            ) from exc

        if (
            isinstance(data, dict)
            and data.get("snapshot_id")
        ):
            snapshot_id = data[
                "snapshot_id"
            ]

            self.log(
                f"Continuing snapshot "
                f"{snapshot_id}"
            )

            return self.wait_for_snapshot(
                snapshot_id,
                timeout_seconds=(
                    timeout_seconds
                ),
            )

        return self.normalize_records(data)

    def trigger_dataset(
        self,
        dataset_id,
        payload,
    ):
        response = requests.post(
            BD_TRIGGER_URL,
            headers=self.headers,
            params={
                "dataset_id": dataset_id,
                "format": "json",
                "include_errors": "true",
            },
            json=payload,
            timeout=60,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"Snapshot trigger failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:2000]}"
            )

        data = response.json()

        snapshot_id = (
            data.get("snapshot_id")
            if isinstance(data, dict)
            else None
        )

        if not snapshot_id:
            raise BrightDataAPIError(
                "Snapshot trigger did not "
                "return snapshot_id."
            )

        return snapshot_id

    def google_ai_mode(
        self,
        prompt,
        timeout_seconds=720,
    ):
        payload = {
            "input": [
                {
                    "url": (
                        "https://google.com/"
                        "aimode"
                    ),
                    "prompt": prompt,
                    "country": self.country,
                }
            ]
        }

        records = self.scrape_dataset(
            dataset_id=(
                GOOGLE_AI_MODE_DATASET_ID
            ),
            payload=payload,
            timeout_seconds=(
                timeout_seconds
            ),
            custom_output_fields=(
                GOOGLE_AI_OUTPUT_FIELDS
            ),
        )

        for record in records:
            if self.answer_text(record):
                return record

        raise BrightDataAPIError(
            "Google AI Mode returned "
            "no answer text."
        )

    def google_serp(
        self,
        query,
        language="en",
        num_results=10,
    ):
        search_url = (
            "https://www.google.com/"
            "search"
            f"?q={quote_plus(query)}"
            f"&gl={self.country.lower()}"
            f"&hl={language.lower()}"
            f"&num={num_results}"
        )

        response = requests.post(
            BD_REQUEST_URL,
            headers=self.headers,
            json={
                "zone": self.serp_zone,
                "url": search_url,
                "format": "raw",
                "data_format": (
                    "parsed_light"
                ),
            },
            timeout=90,
        )

        if not response.ok:
            raise BrightDataAPIError(
                f"SERP request failed. "
                f"HTTP {response.status_code}: "
                f"{response.text[:1500]}"
            )

        data = response.json()
        organic = data.get(
            "organic",
            [],
        )

        results = []

        for position, item in enumerate(
            organic,
            start=1,
        ):
            result_url = (
                item.get("link")
                or item.get("url")
                or ""
            )

            results.append(
                {
                    "rank": (
                        item.get("rank")
                        or position
                    ),
                    "title": item.get(
                        "title",
                        "",
                    ),
                    "url": result_url,
                    "domain": get_hostname(
                        result_url
                    ),
                    "description": (
                        item.get(
                            "description"
                        )
                        or item.get(
                            "snippet"
                        )
                        or ""
                    ),
                }
            )

        return {
            "query": query,
            "results": results,
        }

    def _engine_payload(
        self,
        engine,
        prompt,
        request_index,
        web_search=True,
    ):
        if engine == "chatgpt":
            item = {
                "url": (
                    "https://chatgpt.com/"
                ),
                "prompt": prompt,
                "country": self.country,
                "index": request_index,
                "web_search": web_search,
            }

            return (
                CHATGPT_DATASET_ID,
                [item],
            )

        if engine == "gemini":
            item = {
                "url": (
                    "https://gemini.google.com/"
                ),
                "prompt": prompt,
                "country": self.country,
                "index": request_index,
            }

            return (
                GEMINI_DATASET_ID,
                {"input": [item]},
            )

        raise ValueError(
            f"Unknown AI engine: {engine}"
        )

    def race_ai_engine(
        self,
        engine,
        prompt,
        redundancy=3,
        timeout_seconds=600,
    ):
        engine_name = (
            "ChatGPT"
            if engine == "chatgpt"
            else "Gemini"
        )

        started_at = time.monotonic()

        def trigger_one(index):
            dataset_id, payload = (
                self._engine_payload(
                    engine=engine,
                    prompt=prompt,
                    request_index=index,
                    web_search=True,
                )
            )

            return {
                "request_index": index,
                "snapshot_id": (
                    self.trigger_dataset(
                        dataset_id,
                        payload,
                    )
                ),
            }

        trigger_results = []

        with ThreadPoolExecutor(
            max_workers=redundancy
        ) as executor:
            futures = [
                executor.submit(
                    trigger_one,
                    index,
                )
                for index in range(
                    1,
                    redundancy + 1,
                )
            ]

            for future in as_completed(
                futures
            ):
                try:
                    trigger_results.append(
                        future.result()
                    )
                except Exception as exc:
                    self.log(
                        f"{engine_name} trigger "
                        f"failed: {exc}",
                        "yellow",
                    )

        if not trigger_results:
            raise BrightDataAPIError(
                f"All {engine_name} "
                f"triggers failed."
            )

        invalid_snapshots = set()
        failed_snapshots = set()

        while True:
            elapsed = (
                time.monotonic()
                - started_at
            )

            if elapsed >= timeout_seconds:
                raise TimeoutError(
                    f"No valid {engine_name} "
                    f"snapshot became ready "
                    f"within {timeout_seconds}s."
                )

            for item in trigger_results:
                snapshot_id = item[
                    "snapshot_id"
                ]

                if (
                    snapshot_id
                    in invalid_snapshots
                    or snapshot_id
                    in failed_snapshots
                ):
                    continue

                status = self.snapshot_status(
                    snapshot_id
                )["status"]

                if status in FAILED_STATUSES:
                    failed_snapshots.add(
                        snapshot_id
                    )
                    continue

                if status != "ready":
                    continue

                records = (
                    self.download_snapshot(
                        snapshot_id
                    )
                )

                for record in records:
                    answer = self.answer_text(
                        record
                    )

                    if not answer:
                        continue

                    return {
                        "engine": engine,
                        "engine_name": (
                            engine_name
                        ),
                        "status": "success",
                        "winner_snapshot_id": (
                            snapshot_id
                        ),
                        "winner_request_index": (
                            item[
                                "request_index"
                            ]
                        ),
                        "all_snapshot_ids": [
                            trigger[
                                "snapshot_id"
                            ]
                            for trigger in (
                                trigger_results
                            )
                        ],
                        "duration_seconds": (
                            round(
                                elapsed,
                                2,
                            )
                        ),
                        "answer": answer,
                        "record": record,
                        "citations": (
                            record.get(
                                "citations",
                                [],
                            )
                            or record.get(
                                "search_sources",
                                [],
                            )
                            or []
                        ),
                        "web_search_triggered": (
                            record.get(
                                "web_search_triggered"
                            )
                        ),
                    }

                invalid_snapshots.add(
                    snapshot_id
                )

            if (
                len(failed_snapshots)
                + len(invalid_snapshots)
                >= len(trigger_results)
            ):
                raise BrightDataAPIError(
                    f"All {engine_name} "
                    f"snapshots failed or "
                    f"returned no answer."
                )

            time.sleep(5)

    def generate_chatgpt_report(
        self,
        prompt,
        timeout_seconds=600,
    ):
        item = {
            "url": "https://chatgpt.com/",
            "prompt": prompt,
            "country": self.country,
            "web_search": False,
        }

        # The array request shape has been
        # validated by the working notebook.
        snapshot_id = self.trigger_dataset(
            CHATGPT_DATASET_ID,
            [item],
        )

        records = self.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=(
                timeout_seconds
            ),
        )

        for record in records:
            answer = self.answer_text(
                record
            )

            if answer:
                return {
                    "snapshot_id": (
                        snapshot_id
                    ),
                    "answer": (
                        remove_ai_boilerplate(
                            answer
                        )
                    ),
                    "record": record,
                }

        raise BrightDataAPIError(
            "Final ChatGPT snapshot "
            "returned no report text."
        )


# ============================================================
# SERP competitor helpers
# ============================================================

NON_COMPETITOR_DOMAINS = {
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "twitter.com",
    "x.com",
    "youtube.com",
    "tiktok.com",
    "pinterest.com",
    "reddit.com",
    "quora.com",
    "wikipedia.org",
    "stackoverflow.com",
    "stackexchange.com",
    "g2.com",
    "capterra.com",
    "trustradius.com",
    "trustpilot.com",
    "getapp.com",
    "softwareadvice.com",
    "sourceforge.net",
    "alternativeto.net",
    "saasworthy.com",
    "crunchbase.com",
    "zoominfo.com",
    "bloomberg.com",
    "pitchbook.com",
    "glassdoor.com",
    "indeed.com",
    "medium.com",
    "substack.com",
    "dev.to",
    "forbes.com",
    "techcrunch.com",
    "businessinsider.com",
    "zdnet.com",
    "venturebeat.com",
    "gartner.com",
    "forrester.com",
    "coursera.org",
    "udemy.com",
    "researchgate.net",
    "arxiv.org",
}


def is_non_competitor_domain(
    domain,
):
    root = get_root_domain(domain)

    if not root:
        return True

    return any(
        root == excluded
        or root.endswith(
            f".{excluded}"
        )
        for excluded
        in NON_COMPETITOR_DOMAINS
    )


def looks_like_irrelevant_result(
    result,
):
    result_url = str(
        result.get("url", "")
    ).lower()

    title = str(
        result.get("title", "")
    ).lower()

    url_patterns = {
        "/jobs/",
        "/careers/",
        "/job/",
        "/news/",
        "/press/",
        "/events/",
        "/webinar/",
        "/podcast/",
    }

    title_patterns = {
        "salary",
        "jobs at",
        "careers at",
        "interview questions",
    }

    return (
        any(
            pattern in result_url
            for pattern in url_patterns
        )
        or any(
            pattern in title
            for pattern in title_patterns
        )
    )


def preferred_homepage_url(
    root_domain,
    hostnames,
):
    """
    Choose a likely official product or company hostname without
    hardcoding any vendor or industry.
    """
    host_counts = Counter(
        hostname
        for hostname in hostnames
        if hostname
    )

    preferred_hostname = (
        host_counts.most_common(1)[0][0]
        if host_counts
        else root_domain
    )

    ignored_subdomains = {
        "blog",
        "blogs",
        "docs",
        "documentation",
        "developer",
        "developers",
        "help",
        "support",
        "community",
        "forum",
        "forums",
        "news",
        "careers",
        "jobs",
    }

    first_label = (
        preferred_hostname
        .split(".")[0]
        .lower()
        if preferred_hostname
        else ""
    )

    # Preserve a meaningful product subdomain when it is the hostname
    # that actually ranks. Otherwise use the root domain.
    if (
        preferred_hostname
        and preferred_hostname.endswith(
            root_domain
        )
        and first_label
        not in ignored_subdomains
    ):
        homepage_hostname = (
            preferred_hostname
        )
    else:
        homepage_hostname = (
            root_domain
        )

    return (
        homepage_hostname,
        f"https://{homepage_hostname}/",
    )



def aggregate_competitor_domains(
    keyword_serp_results,
    target_domain,
    total_keyword_count,
):
    target_root = get_root_domain(
        target_domain
    )

    domain_data = defaultdict(
        lambda: {
            "ranks": [],
            "keywords": set(),
            "urls": [],
            "titles": [],
            "hostnames": [],
        }
    )

    for keyword_result in (
        keyword_serp_results
    ):
        if not keyword_result.get(
            "success"
        ):
            continue

        keyword = keyword_result[
            "keyword"
        ]

        seen_for_keyword = set()

        for position, result in enumerate(
            keyword_result.get(
                "results",
                [],
            ),
            start=1,
        ):
            hostname = (
                get_hostname(
                    result.get(
                        "url",
                        "",
                    )
                )
                or result.get(
                    "domain",
                    "",
                )
            )

            root = get_root_domain(
                hostname
            )

            if (
                not root
                or root == target_root
                or root in seen_for_keyword
                or is_non_competitor_domain(
                    root
                )
                or looks_like_irrelevant_result(
                    result
                )
            ):
                continue

            seen_for_keyword.add(root)

            try:
                rank = int(
                    result.get(
                        "rank",
                        position,
                    )
                )
            except Exception:
                rank = position

            domain_data[root][
                "ranks"
            ].append(rank)

            domain_data[root][
                "keywords"
            ].add(keyword)

            domain_data[root][
                "urls"
            ].append(
                result.get("url", "")
            )

            domain_data[root][
                "titles"
            ].append(
                result.get("title", "")
            )

            domain_data[root][
                "hostnames"
            ].append(hostname)

    candidates = []

    for domain, data in (
        domain_data.items()
    ):
        ranks = data["ranks"]

        if not ranks:
            continue

        frequency = len(
            data["keywords"]
        )

        rank_score = sum(
            1 / max(rank, 1)
            for rank in ranks
        )

        preferred_hostname, homepage = (
            preferred_homepage_url(
                domain,
                data["hostnames"],
            )
        )

        total_score = (
            frequency * 100
            + rank_score * 25
            + max(
                0,
                11 - min(ranks),
            )
        )

        candidates.append(
            CompetitorCandidate(
                domain=domain,
                preferred_hostname=(
                    preferred_hostname
                ),
                homepage_url=homepage,
                frequency=frequency,
                keyword_coverage=round(
                    frequency
                    / total_keyword_count,
                    4,
                ),
                best_rank=min(ranks),
                average_rank=round(
                    sum(ranks)
                    / len(ranks),
                    2,
                ),
                rank_score=round(
                    rank_score,
                    4,
                ),
                total_score=round(
                    total_score,
                    2,
                ),
                matched_keywords=sorted(
                    data["keywords"]
                ),
                serp_urls=[
                    url
                    for url in data["urls"]
                    if url
                ],
                serp_titles=[
                    title
                    for title in (
                        data["titles"]
                    )
                    if title
                ],
            )
        )

    candidates.sort(
        key=lambda item: (
            -item.frequency,
            -item.total_score,
            item.average_rank,
            item.domain,
        )
    )

    return candidates


# ============================================================
# Initialize client
# ============================================================

bd_client = BrightDataClient(
    token=BRIGHTDATA_API_TOKEN,
    serp_zone=SERP_ZONE,
    country=COUNTRY,
    debug=DEBUG_MODE,
)


# ============================================================
# Integrated reliability improvements
# ============================================================

def decode_bright_data_response(
    response,
    context,
):
    """
    Decode JSON, JSON text, or NDJSON from a Bright Data response.
    """
    text = (
        response.text
        if response is not None
        else ""
    )

    text = str(
        text or ""
    ).strip()

    if not text:
        raise BrightDataAPIError(
            f"{context} returned an empty response. "
            f"HTTP status: "
            f"{getattr(response, 'status_code', 'unknown')}"
        )

    try:
        return response.json()
    except Exception:
        pass

    try:
        return json.loads(text)
    except Exception:
        pass

    # Some endpoints can return newline-delimited JSON.
    ndjson_records = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        try:
            ndjson_records.append(
                json.loads(line)
            )
        except Exception:
            ndjson_records = []
            break

    if ndjson_records:
        return ndjson_records

    raise BrightDataAPIError(
        f"{context} did not return valid JSON. "
        f"HTTP status: "
        f"{getattr(response, 'status_code', 'unknown')}. "
        f"Response preview: {text[:1000]}"
    )


def parse_ai_json(text):
    """
    Parse JSON-like AI output without leaking JSONDecodeError.

    Supports:
    - Plain JSON
    - Markdown-fenced JSON
    - Google AI Mode Markdown escaping
    - Introductory text surrounding JSON
    - Repairable malformed JSON
    """
    original_text = str(
        text or ""
    ).strip()

    if not original_text:
        raise ValueError(
            "AI returned empty answer text."
        )

    cleaned = clean_ai_json_text(
        original_text
    )

    candidate_strings = [
        cleaned
    ]

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if (
        first_brace >= 0
        and last_brace > first_brace
    ):
        candidate_strings.insert(
            0,
            cleaned[
                first_brace:last_brace + 1
            ],
        )

    errors = []

    for candidate in candidate_strings:
        if not candidate.strip():
            continue

        # Standard JSON.
        try:
            parsed = json.loads(
                candidate
            )

            if isinstance(parsed, dict):
                return parsed

            errors.append(
                "Standard JSON result was "
                f"{type(parsed).__name__}, not an object."
            )

        except Exception as exc:
            errors.append(
                f"Standard JSON: {exc}"
            )

        # JSON repair.
        try:
            repaired = repair_json(
                candidate,
                return_objects=True,
            )

            if isinstance(repaired, dict):
                return repaired

            if isinstance(repaired, list):
                for item in repaired:
                    if isinstance(item, dict):
                        return item

            if (
                isinstance(repaired, str)
                and repaired.strip()
            ):
                try:
                    reparsed = json.loads(
                        repaired
                    )

                    if isinstance(
                        reparsed,
                        dict,
                    ):
                        return reparsed

                except Exception as exc:
                    errors.append(
                        f"Repaired string JSON: {exc}"
                    )

        except Exception as exc:
            errors.append(
                f"JSON repair: {exc}"
            )

    raise ValueError(
        "AI response could not be parsed as a JSON object.\n\n"
        f"Response preview:\n{original_text[:2500]}\n\n"
        f"Parser errors:\n- "
        + "\n- ".join(errors[-6:])
    )


def reliable_snapshot_status(
    self,
    snapshot_id,
):
    """
    Snapshot status that treats temporary empty/non-JSON responses as
    unknown rather than crashing the audit.
    """
    try:
        response = requests.get(
            f"{BD_PROGRESS_URL}/"
            f"{snapshot_id}",
            headers=self.headers,
            timeout=30,
        )

        if not response.ok:
            return {
                "status": "unknown",
                "details": (
                    response.text[:1000]
                ),
            }

        data = decode_bright_data_response(
            response,
            context=(
                f"Progress endpoint for "
                f"{snapshot_id}"
            ),
        )

        if not isinstance(data, dict):
            return {
                "status": "unknown",
                "details": data,
            }

        return {
            "status": str(
                data.get(
                    "status",
                    "unknown",
                )
            ).lower(),
            "details": data,
        }

    except Exception as exc:
        self.log(
            f"Temporary progress error for "
            f"{snapshot_id}: {exc}",
            "yellow",
        )

        return {
            "status": "unknown",
            "details": str(exc),
        }


def reliable_download_snapshot(
    self,
    snapshot_id,
    attempts=4,
):
    """
    Retry temporarily empty or non-JSON snapshot downloads.
    """
    last_error = None

    for attempt in range(
        1,
        attempts + 1,
    ):
        try:
            response = requests.get(
                f"{BD_SNAPSHOT_URL}/"
                f"{snapshot_id}",
                headers=self.headers,
                params={"format": "json"},
                timeout=90,
            )

            if not response.ok:
                raise BrightDataAPIError(
                    f"Snapshot download failed. "
                    f"HTTP {response.status_code}: "
                    f"{response.text[:1500]}"
                )

            data = (
                decode_bright_data_response(
                    response,
                    context=(
                        f"Snapshot download "
                        f"{snapshot_id}"
                    ),
                )
            )

            records = (
                self.normalize_records(
                    data
                )
            )

            if not records:
                raise BrightDataAPIError(
                    "Snapshot download returned "
                    "no records."
                )

            return records

        except Exception as exc:
            last_error = exc

            self.log(
                f"Snapshot download attempt "
                f"{attempt}/{attempts} failed "
                f"for {snapshot_id}: {exc}",
                "yellow",
            )

            if attempt < attempts:
                time.sleep(
                    attempt * 3
                )

    raise BrightDataAPIError(
        f"Could not download snapshot "
        f"{snapshot_id} after "
        f"{attempts} attempts: "
        f"{last_error}"
    )


def reliable_trigger_dataset(
    self,
    dataset_id,
    payload,
):
    response = requests.post(
        BD_TRIGGER_URL,
        headers=self.headers,
        params={
            "dataset_id": dataset_id,
            "format": "json",
            "include_errors": "true",
        },
        json=payload,
        timeout=60,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"Snapshot trigger failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"Snapshot trigger for "
            f"{dataset_id}"
        ),
    )

    snapshot_id = (
        data.get("snapshot_id")
        if isinstance(data, dict)
        else None
    )

    if not snapshot_id:
        raise BrightDataAPIError(
            "Snapshot trigger did not "
            f"return snapshot_id: {data}"
        )

    return snapshot_id


def reliable_scrape_dataset(
    self,
    dataset_id,
    payload,
    timeout_seconds=600,
    custom_output_fields=None,
):
    params = {
        "dataset_id": dataset_id,
        "format": "json",
        "notify": "false",
        "include_errors": "true",
    }

    if custom_output_fields:
        params[
            "custom_output_fields"
        ] = custom_output_fields

    response = requests.post(
        BD_SCRAPE_URL,
        headers=self.headers,
        params=params,
        json=payload,
        timeout=90,
    )

    if response.status_code not in {
        200,
        202,
    }:
        raise BrightDataAPIError(
            f"Dataset request failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:2000]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"Dataset request for "
            f"{dataset_id}"
        ),
    )

    if (
        isinstance(data, dict)
        and data.get("snapshot_id")
    ):
        snapshot_id = data[
            "snapshot_id"
        ]

        self.log(
            f"Continuing snapshot "
            f"{snapshot_id}"
        )

        return self.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=(
                timeout_seconds
            ),
        )

    return self.normalize_records(data)


def remove_ai_boilerplate(text):
    """
    Remove common ChatGPT/Gemini UI and conversational boilerplate.
    """
    if not isinstance(text, str):
        return ""

    cleaned_lines = []

    exact_unwanted = {
        "log in",
        "login",
        "sign up",
        "sign up for free",
        "log insign up for free",
    }

    unwanted_prefixes = (
        "log in for more personalized",
        "if you want, i can",
        "would you like me to",
        "below is a professional audit",
        "below is the requested audit",
        "here is the requested audit",
    )

    for line in text.splitlines():
        normalized = re.sub(
            r"\s+",
            " ",
            line,
        ).strip().lower()

        if normalized in exact_unwanted:
            continue

        if any(
            normalized.startswith(prefix)
            for prefix in unwanted_prefixes
        ):
            continue

        cleaned_lines.append(line)

    return "\n".join(
        cleaned_lines
    ).strip()


# Use the reliable implementations directly.
BrightDataClient.snapshot_status = (
    reliable_snapshot_status
)

BrightDataClient.download_snapshot = (
    reliable_download_snapshot
)

BrightDataClient.trigger_dataset = (
    reliable_trigger_dataset
)

BrightDataClient.scrape_dataset = (
    reliable_scrape_dataset
)


console.print(
    "[bold green]✓ Core audit engine loaded[/bold green]"
)
console.print(
    f"Country: {bd_client.country}"
)
console.print(
    f"Debug logging: "
    f"{'enabled' if bd_client.debug else 'disabled'}"
)


# ============================================================
# Flexible AI-answer and SERP normalization
# ============================================================

def reliable_answer_text(
    record,
):
    """
    Choose a sensible answer representation.

    Prefer Markdown for normal answers, but prefer plain text when the
    Markdown field is unexpectedly huge.
    """
    if not isinstance(record, dict):
        return str(record or "").strip()

    plain_text = (
        record.get("answer_text")
        or record.get("answer")
        or record.get("response")
        or record.get("text")
        or ""
    )

    markdown_text = (
        record.get("answer_text_markdown")
        or record.get("answer_markdown")
        or ""
    )

    if not isinstance(
        plain_text,
        str,
    ):
        plain_text = (
            json.dumps(
                plain_text,
                ensure_ascii=False,
            )
            if plain_text
            else ""
        )

    if not isinstance(
        markdown_text,
        str,
    ):
        markdown_text = (
            json.dumps(
                markdown_text,
                ensure_ascii=False,
            )
            if markdown_text
            else ""
        )

    plain_text = plain_text.strip()
    markdown_text = markdown_text.strip()

    # Some answer-engine records occasionally contain an extremely
    # large Markdown capture. Prefer the concise plain answer.
    if (
        len(markdown_text) > 100_000
        and plain_text
    ):
        return plain_text

    if markdown_text:
        return markdown_text

    return plain_text


def extract_serp_url_value(
    item,
):
    """
    Extract a usable result URL from different parsed SERP schemas.
    """
    from urllib.parse import (
        parse_qs,
        unquote,
        urlparse,
    )

    if not isinstance(item, dict):
        return ""

    preferred_fields = (
        "link",
        "url",
        "href",
        "result_url",
        "target_url",
        "redirect_url",
    )

    value = ""

    for field_name in preferred_fields:
        candidate = item.get(
            field_name
        )

        if isinstance(candidate, dict):
            candidate = (
                candidate.get("url")
                or candidate.get("link")
                or candidate.get("href")
                or ""
            )

        if isinstance(candidate, str):
            candidate = candidate.strip()

            if candidate:
                value = candidate
                break

    # As a last resort, search the record for a normal HTTP URL.
    if not value:
        for candidate in item.values():
            if (
                isinstance(candidate, str)
                and "http" in candidate
            ):
                match = re.search(
                    r"https?://[^\s\"'<>]+",
                    candidate,
                )

                if match:
                    value = match.group(0)
                    break

    if not value:
        return ""

    # Protocol-relative URL.
    if value.startswith("//"):
        value = f"https:{value}"

    # Google redirect URL.
    if value.startswith("/"):
        parsed_relative = urlparse(
            value
        )

        query = parse_qs(
            parsed_relative.query
        )

        redirected = (
            query.get("q", [""])[0]
            or query.get(
                "url",
                [""],
            )[0]
        )

        if redirected:
            value = unquote(
                redirected
            )
        else:
            return ""

    # A plain hostname can still be useful.
    if not value.lower().startswith(
        ("http://", "https://")
    ):
        if (
            "." in value
            and " " not in value
        ):
            value = (
                f"https://{value}"
            )
        else:
            return ""

    return value


def extract_serp_domain_value(
    item,
    result_url="",
):
    """
    Extract a root domain from a SERP result using multiple fallbacks.
    """
    domain = get_root_domain(
        result_url
    )

    if domain:
        return domain

    if not isinstance(item, dict):
        return ""

    domain_fields = (
        "domain",
        "displayed_link",
        "display_link",
        "visible_url",
        "source",
        "site_name",
        "hostname",
    )

    for field_name in domain_fields:
        candidate = item.get(
            field_name
        )

        if isinstance(candidate, dict):
            candidate = (
                candidate.get("domain")
                or candidate.get("url")
                or candidate.get("name")
                or ""
            )

        if not isinstance(
            candidate,
            str,
        ):
            continue

        candidate = (
            candidate.strip()
            .replace("›", "/")
        )

        if not candidate:
            continue

        # Keep only the first URL/domain-looking component.
        candidate = candidate.split()[0]
        candidate = candidate.split("/")[0]

        domain = get_root_domain(
            candidate
        )

        if domain:
            return domain

    return ""


def reliable_google_serp(
    self,
    query,
    language="en",
    num_results=10,
):
    """
    Query SERP API and normalize several possible parsed-result shapes.
    """
    search_url = (
        "https://www.google.com/search"
        f"?q={quote_plus(query)}"
        f"&gl={self.country.lower()}"
        f"&hl={language.lower()}"
        f"&num={num_results}"
    )

    response = requests.post(
        BD_REQUEST_URL,
        headers=self.headers,
        json={
            "zone": self.serp_zone,
            "url": search_url,
            "format": "raw",
            "data_format": (
                "parsed_light"
            ),
        },
        timeout=90,
    )

    if not response.ok:
        raise BrightDataAPIError(
            f"SERP request failed. "
            f"HTTP {response.status_code}: "
            f"{response.text[:1500]}"
        )

    data = decode_bright_data_response(
        response,
        context=(
            f"SERP request for {query!r}"
        ),
    )

    if not isinstance(data, dict):
        raise BrightDataAPIError(
            "SERP response was not a JSON object."
        )

    organic = (
        data.get("organic")
        or data.get("results")
        or data.get(
            "organic_results"
        )
        or []
    )

    if not isinstance(organic, list):
        organic = []

    results = []

    for position, item in enumerate(
        organic,
        start=1,
    ):
        if not isinstance(item, dict):
            continue

        result_url = (
            extract_serp_url_value(
                item
            )
        )

        domain = (
            extract_serp_domain_value(
                item,
                result_url,
            )
        )

        if (
            not result_url
            and domain
        ):
            result_url = (
                f"https://{domain}/"
            )

        if self.debug and not domain:
            self.log(
                f"SERP result had no domain. "
                f"Query={query!r}; "
                f"fields={list(item.keys())}; "
                f"record={str(item)[:500]}",
                "yellow",
            )

        results.append(
            {
                "rank": (
                    item.get("rank")
                    or item.get("position")
                    or position
                ),
                "title": (
                    item.get("title")
                    or item.get("name")
                    or ""
                ),
                "url": result_url,
                "domain": domain,
                "description": (
                    item.get("description")
                    or item.get("snippet")
                    or item.get("text")
                    or ""
                ),
            }
        )

    return {
        "query": query,
        "results": results,
        "raw_result_count": len(
            organic
        ),
    }


BrightDataClient.answer_text = staticmethod(
    reliable_answer_text
)

BrightDataClient.google_serp = (
    reliable_google_serp
)


✓ Core audit engine loaded

Country: US

Debug logging: enabled

In [5]:
#@title 3B. Load analysis pipeline
#@markdown Google AI Mode research, ChatGPT structuring, parallel SERPs, competitor selection, and profiles.
#@markdown This cell normally does not need to be edited.

# ============================================================
# Input normalization
# ============================================================

def normalize_keyword_records(
    raw_keywords,
):
    if isinstance(raw_keywords, str):
        raw_keywords = [
            item.strip()
            for item in raw_keywords.split(",")
            if item.strip()
        ]

    if not isinstance(raw_keywords, list):
        return []

    normalized = []
    seen = set()

    for item in raw_keywords:
        if isinstance(item, str):
            keyword = item.strip()

            record = {
                "keyword": keyword,
                "intent": "commercial",
                "rationale": "",
            }

        elif isinstance(item, dict):
            keyword = str(
                item.get("keyword")
                or item.get("query")
                or item.get("term")
                or ""
            ).strip()

            record = {
                "keyword": keyword,
                "intent": str(
                    item.get("intent")
                    or "commercial"
                ).strip(),
                "rationale": str(
                    item.get("rationale")
                    or item.get("reason")
                    or ""
                ).strip(),
            }

        else:
            continue

        key = keyword.lower()

        if not key or key in seen:
            continue

        seen.add(key)
        normalized.append(record)

    return normalized


def normalize_company_intake(
    data,
    company_name,
    company_url,
):
    if not isinstance(data, dict):
        raise ValueError(
            "Company analysis must be a JSON object."
        )

    if "brand" not in data:
        brand_keys = {
            "brand_name",
            "official_url",
            "domain",
            "category",
            "description",
            "positioning",
            "target_customers",
            "products",
            "key_features",
            "differentiators",
            "confidence",
            "evidence",
        }

        data["brand"] = {
            key: data[key]
            for key in data
            if key in brand_keys
        }

    brand = data.get("brand") or {}

    if not isinstance(brand, dict):
        brand = {}

    official_url = normalize_public_url(
        brand.get("official_url")
        or company_url
    )

    if not official_url:
        official_url = normalize_public_url(
            company_url
        )

    domain = get_root_domain(
        brand.get("domain")
        or official_url
    )

    brand["brand_name"] = str(
        brand.get("brand_name")
        or company_name
    ).strip()

    if not brand["brand_name"]:
        brand["brand_name"] = company_name

    brand["official_url"] = official_url
    brand["domain"] = domain

    for field_name in (
        "category",
        "description",
        "positioning",
    ):
        brand[field_name] = str(
            brand.get(field_name)
            or ""
        ).strip()

    for field_name in (
        "target_customers",
        "products",
        "key_features",
        "differentiators",
        "evidence",
    ):
        brand[field_name] = (
            ensure_string_list(
                brand.get(field_name)
            )[:8]
        )

    brand["confidence"] = (
        normalize_confidence(
            brand.get("confidence")
        )
    )

    keywords = normalize_keyword_records(
        data.get("buyer_intent_keywords")
        or data.get("keywords")
        or []
    )

    normalized = {
        "brand": brand,
        "buyer_intent_keywords": keywords,
    }

    return validate_model(
        CompanyIntake,
        normalized,
    )


# ============================================================
# Reliable company-analysis workflow
# ============================================================

def run_chatgpt_without_web(
    prompt,
    timeout_seconds=900,
):
    """
    Run one ChatGPT snapshot with web search disabled.

    The longer timeout avoids abandoning a valid structuring snapshot
    shortly before it completes.
    """
    if len(prompt) > 4096:
        raise ValueError(
            f"ChatGPT transformation prompt is too long: "
            f"{len(prompt)} characters."
        )

    item = {
        "url": "https://chatgpt.com/",
        "prompt": prompt,
        "country": bd_client.country,
        "web_search": False,
    }

    snapshot_id = (
        bd_client.trigger_dataset(
            CHATGPT_DATASET_ID,
            [item],
        )
    )

    bd_client.log(
        f"ChatGPT transformation snapshot: "
        f"{snapshot_id}"
    )

    records = (
        bd_client.wait_for_snapshot(
            snapshot_id,
            timeout_seconds=timeout_seconds,
        )
    )

    for record in records:
        answer = (
            bd_client.answer_text(
                record
            )
        )

        if answer:
            return {
                "snapshot_id": (
                    snapshot_id
                ),
                "record": record,
                "answer": answer,
            }

    raise BrightDataAPIError(
        "ChatGPT transformation snapshot "
        "returned no answer text."
    )



def build_company_research_prompt(
    settings,
):
    """
    Ask Google AI Mode to research any company, brand, product,
    service, institution, or local business.
    """
    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    focus_instruction = (
        f"Specific audit focus: {audit_focus}"
        if audit_focus
        else (
            "Audit focus: infer the primary product, service, "
            "offering, or customer need represented by the website."
        )
    )

    return f"""
Analyze the current public website for this organization or brand.

Name: {settings["company_name"]}
Website: {settings["company_url"]}
Country: {settings["country"]}
{focus_instruction}

First determine what kind of market this is, such as consumer product,
B2B product, software, professional service, local business,
health/beauty product, retailer, financial product, education,
hospitality, or another category.

Provide a concise research brief covering:

- Canonical brand, organization, or product name
- The specific product, service, or offering being audited
- Market and product category
- Intended customer, user, or audience
- Primary customer need or problem addressed
- Main products, services, or alternatives offered
- Important features, benefits, claims, or capabilities
- Relevant proof, trust signals, ingredients, specifications, or
  evidence when applicable
- Meaningful differentiators
- Price, price tier, pricing approach, or availability when public
- The criteria a real customer would use to compare alternatives
- Exactly eight non-branded buyer-intent searches that could be used
  to find this offering and competing alternatives

The buyer searches must match the actual market. They may be consumer,
commercial, transactional, local, or solution-evaluation searches.

Do not include the brand name or competitor names in the searches.

Use current public information. Keep the response concise. Do not ask
follow-up questions.
""".strip()



def build_company_structuring_prompt(
    settings,
    research_text,
    strict_retry=False,
):
    """
    Build a ChatGPT structuring prompt that dynamically fits below the
    input limit.
    """
    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    retry_instruction = ""

    if strict_retry:
        retry_instruction = """
A previous formatting attempt failed. Return the JSON object directly.
The first character must be { and the final character must be }.
Do not include a code fence or explanatory sentence.
""".strip()

    template = """
Convert the supplied Google AI Mode research into structured JSON.

Do not perform new web research. Use only the supplied research and
the known name, website, and optional audit focus.

Known name: {company_name}
Known website: {company_url}
Audit focus: {audit_focus}

GOOGLE AI MODE RESEARCH
-----------------------
{research_text}
-----------------------
END RESEARCH

Interpret the fields generically:

- products can contain products, services, plans, treatments,
  experiences, courses, or other offerings
- key_features can contain features, benefits, ingredients, claims,
  specifications, or service attributes
- target_customers can contain buyers, users, audiences, patients,
  guests, students, or business segments
- differentiators can contain meaningful reasons a customer might
  choose the offering over an alternative

Return exactly this JSON structure:

{{
  "brand": {{
    "brand_name": "canonical name",
    "official_url": "{company_url}",
    "domain": "{company_domain}",
    "category": "specific market or offering category",
    "description": "one or two sentences",
    "positioning": "customer-facing positioning",
    "target_customers": ["audience or customer"],
    "products": ["product, service, or offering"],
    "key_features": ["feature, benefit, claim, or attribute"],
    "differentiators": ["meaningful differentiator"],
    "confidence": 0.0,
    "evidence": ["evidence from the supplied research"]
  }},
  "buyer_intent_keywords": [
    {{
      "keyword": "non-branded buyer query",
      "intent": "commercial",
      "rationale": "short reason"
    }}
  ]
}}

Requirements:
- Return exactly eight unique keyword objects.
- Keywords must be natural for this specific market.
- Do not use the audited name or competitor names in a keyword.
- Confidence must be between 0 and 1.
- Use the known website and domain when research contains a redirect.
- Do not invent facts absent from the research.
- Return JSON only without Markdown fences.

{retry_instruction}
""".strip()

    research_text = str(
        research_text or ""
    ).strip()

    # Begin with a useful excerpt and reduce it only when necessary.
    research_limit = min(
        len(research_text),
        1700,
    )

    while research_limit >= 600:
        fitted_research = (
            research_text[
                :research_limit
            ]
        )

        prompt = template.format(
            company_name=(
                settings[
                    "company_name"
                ]
            ),
            company_url=(
                settings[
                    "company_url"
                ]
            ),
            company_domain=(
                settings[
                    "company_domain"
                ]
            ),
            audit_focus=(
                audit_focus
                or "primary offering"
            ),
            research_text=(
                fitted_research
            ),
            retry_instruction=(
                retry_instruction
            ),
        )

        if len(prompt) <= 3900:
            if bd_client.debug:
                bd_client.log(
                    f"Company structuring prompt: "
                    f"{len(prompt)} characters; "
                    f"research excerpt: "
                    f"{len(fitted_research)} characters"
                )

            return prompt

        research_limit -= 100

    raise ValueError(
        "Could not fit the company structuring "
        "prompt below 3,900 characters."
    )




def complete_company_keywords(
    settings,
    brand,
    current_keywords,
):
    """
    Complete a buyer-intent keyword set for any product, service,
    organization, or consumer category.
    """
    existing = [
        item.keyword
        for item in current_keywords
    ]

    audit_focus = str(
        settings.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    prompt = f"""
Using only the supplied information, return exactly eight unique
non-branded buyer-intent searches appropriate for this market.

Category: {brand.category}
Audit focus: {audit_focus or "primary offering"}
Positioning: {brand.positioning}
Offerings: {", ".join(brand.products[:5])}
Attributes or benefits: {", ".join(brand.key_features[:6])}

Existing keywords:
{json.dumps(existing, ensure_ascii=False)}

The searches should sound like something a real customer would enter
when looking for, evaluating, comparing, or buying alternatives.

They may be consumer, B2B, local, commercial, transactional, or
solution-evaluation searches depending on the category.

Return only JSON:

{{
  "buyer_intent_keywords": [
    {{
      "keyword": "buyer query",
      "intent": "commercial",
      "rationale": "short reason"
    }}
  ]
}}

Do not include the audited brand or competitor names.
""".strip()

    result = run_chatgpt_without_web(
        prompt
    )

    parsed = parse_ai_json(
        result["answer"]
    )

    completed = normalize_keyword_records(
        parsed.get(
            "buyer_intent_keywords"
        )
        or parsed.get("keywords")
        or []
    )

    combined = []
    seen = set()

    audited_name = (
        settings["company_name"]
        .lower()
        .strip()
    )

    for item in [
        *[
            model_to_dict(keyword)
            for keyword in current_keywords
        ],
        *completed,
    ]:
        keyword = str(
            item.get("keyword")
            or ""
        ).strip()

        key = keyword.lower()

        if not key or key in seen:
            continue

        if audited_name in key:
            continue

        seen.add(key)

        combined.append(
            validate_model(
                BuyerIntentKeyword,
                item,
            )
        )

        if len(combined) == 8:
            break

    return {
        "keywords": combined,
        "record": result["record"],
        "snapshot_id": (
            result["snapshot_id"]
        ),
    }



def analyze_company_stage(
    settings,
):
    """
    Reliable company analysis:

    1. Google AI Mode performs natural-language research.
    2. ChatGPT transforms the research into JSON.
    3. A second ChatGPT formatting attempt runs only if necessary.
    4. Keyword completion runs only if fewer than eight were returned.
    """
    bd_client.log(
        "Starting Google AI Mode company research"
    )

    research_prompt = (
        build_company_research_prompt(
            settings
        )
    )

    research_record = (
        bd_client.google_ai_mode(
            research_prompt,
            timeout_seconds=720,
        )
    )

    research_text = (
        bd_client.answer_text(
            research_record
        )
    )

    if not research_text:
        raise BrightDataAPIError(
            "Google AI Mode returned no "
            "company research text."
        )

    bd_client.log(
        f"Google AI Mode research returned "
        f"{len(research_text):,} characters"
    )

    structuring_errors = []
    structured_result = None
    intake = None

    for attempt in (
        1,
        2,
    ):
        bd_client.log(
            f"Starting ChatGPT structuring "
            f"attempt {attempt}/2"
        )

        structuring_prompt = (
            build_company_structuring_prompt(
                settings=settings,
                research_text=research_text,
                strict_retry=(
                    attempt == 2
                ),
            )
        )

        try:
            candidate_result = (
                run_chatgpt_without_web(
                    structuring_prompt
                )
            )

            parsed = parse_ai_json(
                candidate_result[
                    "answer"
                ]
            )

            candidate_intake = (
                normalize_company_intake(
                    data=parsed,
                    company_name=settings[
                        "company_name"
                    ],
                    company_url=settings[
                        "company_url"
                    ],
                )
            )

            structured_result = (
                candidate_result
            )

            intake = candidate_intake
            break

        except Exception as exc:
            structuring_errors.append(
                f"Attempt {attempt}: "
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            bd_client.log(
                f"ChatGPT structuring attempt "
                f"{attempt} failed: {exc}",
                "yellow",
            )

    if intake is None:
        raise BrightDataAPIError(
            "ChatGPT could not structure the "
            "Google AI Mode research.\n- "
            + "\n- ".join(
                structuring_errors
            )
        )

    keyword_completion = None

    if len(
        intake.buyer_intent_keywords
    ) != 8:
        bd_client.log(
            f"Structured result contained "
            f"{len(intake.buyer_intent_keywords)} "
            f"keywords; completing the set"
        )

        keyword_completion = (
            complete_company_keywords(
                settings=settings,
                brand=intake.brand,
                current_keywords=(
                    intake.buyer_intent_keywords
                ),
            )
        )

        intake.buyer_intent_keywords = (
            keyword_completion[
                "keywords"
            ]
        )

    if len(
        intake.buyer_intent_keywords
    ) != 8:
        raise BrightDataAPIError(
            "The company-analysis workflow "
            f"produced "
            f"{len(intake.buyer_intent_keywords)} "
            f"keywords instead of eight."
        )

    return {
        "intake": intake,

        # Preserve Google AI Mode as the main
        # research record expected by the orchestrator.
        "record": research_record,

        "prompt": research_prompt,

        "research_text": research_text,

        "structuring_record": (
            structured_result[
                "record"
            ]
        ),

        "structuring_snapshot_id": (
            structured_result[
                "snapshot_id"
            ]
        ),

        "keyword_completion": (
            keyword_completion
        ),

        "workflow": (
            "google_ai_research_"
            "chatgpt_structuring"
        ),
    }


# ============================================================
# SERP execution with retries
# ============================================================

async def run_keyword_serp_task(
    keyword,
    semaphore,
    num_results=10,
    max_attempts=3,
):
    """
    Run one SERP request with bounded retries.
    """
    async with semaphore:
        started_at = time.monotonic()
        errors = []

        for attempt in range(
            1,
            max_attempts + 1,
        ):
            try:
                response = await asyncio.to_thread(
                    bd_client.google_serp,
                    keyword,
                    "en",
                    num_results,
                )

                if attempt > 1:
                    bd_client.log(
                        f"SERP retry succeeded for "
                        f"{keyword!r} on attempt "
                        f"{attempt}"
                    )

                return {
                    "keyword": keyword,
                    "success": True,
                    "attempts": attempt,
                    "duration_seconds": round(
                        time.monotonic()
                        - started_at,
                        2,
                    ),
                    "results": response[
                        "results"
                    ],
                    "error": None,
                }

            except Exception as exc:
                errors.append(
                    f"Attempt {attempt}: {exc}"
                )

                bd_client.log(
                    f"SERP attempt "
                    f"{attempt}/{max_attempts} "
                    f"failed for {keyword!r}: "
                    f"{exc}",
                    "yellow",
                )

                if attempt < max_attempts:
                    await asyncio.sleep(
                        attempt * 2
                    )

        return {
            "keyword": keyword,
            "success": False,
            "attempts": max_attempts,
            "duration_seconds": round(
                time.monotonic()
                - started_at,
                2,
            ),
            "results": [],
            "error": " | ".join(errors),
        }


# ============================================================
# Competitor and profile pipeline
# ============================================================

async def run_serp_stage(
    keywords,
    target_domain,
):
    """
    Run buyer-intent SERPs and build competitor candidates.

    Strategy:
    1. Request 20 results per keyword.
    2. Apply the strict competitor filter.
    3. If strict filtering removes everything, retain broader domains
       and let the AI competitor-selection stage classify them.
    """
    semaphore = asyncio.Semaphore(
        min(8, len(keywords))
    )

    tasks = [
        run_keyword_serp_task(
            keyword=keyword,
            semaphore=semaphore,
            num_results=20,
        )
        for keyword in keywords
    ]

    results = await asyncio.gather(
        *tasks
    )

    successful = [
        result
        for result in results
        if result["success"]
    ]

    failed = [
        result
        for result in results
        if not result["success"]
    ]

    if not successful:
        error_details = " | ".join(
            result.get(
                "error",
                "unknown error",
            )
            for result in failed
        )

        raise BrightDataAPIError(
            "All Google SERP requests failed. "
            f"Errors: {error_details}"
        )

    if bd_client.debug:
        for result in results:
            if not result["success"]:
                bd_client.log(
                    f"SERP failed for "
                    f"{result['keyword']!r}: "
                    f"{result.get('error')}",
                    "yellow",
                )
                continue

            result_domains = []

            for serp_item in result[
                "results"
            ]:
                domain = get_root_domain(
                    serp_item.get("domain")
                    or serp_item.get("url")
                    or ""
                )

                if (
                    domain
                    and domain
                    not in result_domains
                ):
                    result_domains.append(
                        domain
                    )

            bd_client.log(
                f"SERP {result['keyword']!r}: "
                f"{len(result['results'])} results; "
                f"domains="
                f"{', '.join(result_domains[:8])}"
            )

    # --------------------------------------------------------
    # Strict filter
    # --------------------------------------------------------

    candidates = (
        aggregate_competitor_domains(
            keyword_serp_results=results,
            target_domain=target_domain,
            total_keyword_count=len(
                keywords
            ),
        )
    )

    filter_mode = "strict"

    # --------------------------------------------------------
    # Relaxed fallback
    # --------------------------------------------------------

    if not candidates:
        filter_mode = "relaxed"

        bd_client.log(
            "Strict competitor filtering removed all domains. "
            "Using relaxed candidate collection and deferring "
            "classification to Google AI Mode.",
            "yellow",
        )

        target_root = get_root_domain(
            target_domain
        )

        always_excluded = {
            "facebook.com",
            "instagram.com",
            "linkedin.com",
            "twitter.com",
            "x.com",
            "youtube.com",
            "tiktok.com",
            "pinterest.com",
            "wikipedia.org",
        }

        domain_data = defaultdict(
            lambda: {
                "ranks": [],
                "keywords": set(),
                "urls": [],
                "titles": [],
                "hostnames": [],
            }
        )

        for keyword_result in results:
            if not keyword_result.get(
                "success"
            ):
                continue

            keyword = keyword_result[
                "keyword"
            ]

            seen_for_keyword = set()

            for position, item in enumerate(
                keyword_result.get(
                    "results",
                    [],
                ),
                start=1,
            ):
                result_url = item.get(
                    "url",
                    "",
                )

                hostname = (
                    get_hostname(
                        result_url
                    )
                    or item.get(
                        "domain",
                        "",
                    )
                )

                root_domain = get_root_domain(
                    hostname
                )

                if not root_domain:
                    continue

                if root_domain == target_root:
                    continue

                if root_domain in (
                    always_excluded
                ):
                    continue

                if root_domain in (
                    seen_for_keyword
                ):
                    continue

                seen_for_keyword.add(
                    root_domain
                )

                try:
                    rank = int(
                        item.get(
                            "rank",
                            position,
                        )
                    )
                except Exception:
                    rank = position

                domain_data[root_domain][
                    "ranks"
                ].append(rank)

                domain_data[root_domain][
                    "keywords"
                ].add(keyword)

                domain_data[root_domain][
                    "urls"
                ].append(
                    result_url
                )

                domain_data[root_domain][
                    "titles"
                ].append(
                    item.get(
                        "title",
                        "",
                    )
                )

                domain_data[root_domain][
                    "hostnames"
                ].append(
                    hostname
                )

        relaxed_candidates = []

        for domain, data in (
            domain_data.items()
        ):
            ranks = data["ranks"]

            if not ranks:
                continue

            frequency = len(
                data["keywords"]
            )

            rank_score = sum(
                1 / max(rank, 1)
                for rank in ranks
            )

            (
                preferred_hostname,
                homepage_url,
            ) = preferred_homepage_url(
                domain,
                data["hostnames"],
            )

            total_score = (
                frequency * 100
                + rank_score * 25
                + max(
                    0,
                    21 - min(ranks),
                )
            )

            relaxed_candidates.append(
                CompetitorCandidate(
                    domain=domain,
                    preferred_hostname=(
                        preferred_hostname
                    ),
                    homepage_url=(
                        homepage_url
                    ),
                    frequency=frequency,
                    keyword_coverage=round(
                        frequency
                        / len(keywords),
                        4,
                    ),
                    best_rank=min(ranks),
                    average_rank=round(
                        sum(ranks)
                        / len(ranks),
                        2,
                    ),
                    rank_score=round(
                        rank_score,
                        4,
                    ),
                    total_score=round(
                        total_score,
                        2,
                    ),
                    matched_keywords=sorted(
                        data["keywords"]
                    ),
                    serp_urls=[
                        url
                        for url in data[
                            "urls"
                        ]
                        if url
                    ],
                    serp_titles=[
                        title
                        for title in data[
                            "titles"
                        ]
                        if title
                    ],
                )
            )

        relaxed_candidates.sort(
            key=lambda candidate: (
                -candidate.frequency,
                -candidate.total_score,
                candidate.average_rank,
                candidate.domain,
            )
        )

        candidates = relaxed_candidates

    if not candidates:
        raw_domains = sorted(
            {
                get_root_domain(
                    item.get("domain")
                    or item.get("url")
                    or ""
                )
                for result in successful
                for item in result.get(
                    "results",
                    [],
                )
                if get_root_domain(
                    item.get("domain")
                    or item.get("url")
                    or ""
                )
            }
        )

        raise ValueError(
            "No competitor candidates could be generated even "
            "with relaxed filtering.\n"
            f"Target domain: {target_domain}\n"
            f"Keywords: {keywords}\n"
            f"Observed domains: {raw_domains[:40]}"
        )

    bd_client.log(
        f"Competitor candidate filter mode: "
        f"{filter_mode}; "
        f"{len(candidates)} candidates retained"
    )

    return {
        "keyword_results": results,
        "candidates": candidates,
        "successful": len(successful),
        "failed": len(failed),
        "filter_mode": filter_mode,
    }



def compact_candidate_data(
    candidates,
    limit,
):
    return [
        {
            "domain": item.domain,
            "url": item.homepage_url,
            "appearances": item.frequency,
            "best_rank": item.best_rank,
            "queries": (
                item.matched_keywords[:3]
            ),
        }
        for item in candidates[:limit]
    ]


def build_competitor_selection_prompt(
    target_brand,
    candidates,
    keywords,
):
    shortlist_size = min(
        12,
        len(candidates),
    )

    while shortlist_size >= 7:
        compact_candidates = (
            compact_candidate_data(
                candidates,
                shortlist_size,
            )
        )

        prompt = f"""
Select the five most direct alternatives or competitors for:

Target: {target_brand.brand_name}
Website: {target_brand.official_url}
Category: {target_brand.category}
Positioning: {shorten(target_brand.positioning, 220)}

Buyer searches:
{json.dumps(keywords, ensure_ascii=False)}

Candidate domains found in Google:
{json.dumps(compact_candidates, ensure_ascii=False)}

Determine what a direct competitor means in this specific market.

A direct competitor should address substantially the same customer
need, buyer intent, product category, service category, use case, or
purchase decision.

Distinguish direct competitors from:

- Retailers or marketplaces selling many brands
- Publishers and news sites
- Review and comparison sites
- Communities and forums
- Distributors
- Directories
- Informational resources
- Companies that are only loosely adjacent

Select only from the supplied candidate domains.

Return only JSON:

{{
  "selected_competitors": [
    {{
      "brand_name": "competing brand, product, service, or organization",
      "domain": "supplied candidate domain",
      "official_url": "normal public URL",
      "reason": "why customers would directly compare it",
      "confidence": 0.0
    }}
  ],
  "rejected_candidates": [
    {{
      "domain": "candidate domain",
      "reason": "retailer, publisher, marketplace, indirect, or other"
    }}
  ]
}}

Return exactly five selected competitors. Do not use Markdown fences.
""".strip()

        if len(prompt) <= 3900:
            return (
                prompt,
                candidates[:shortlist_size],
            )

        shortlist_size -= 1

    raise ValueError(
        "Competitor-selection prompt could not be "
        "reduced below the safe limit."
    )



def normalize_selected_competitors(
    data,
    shortlist,
    required_count=5,
):
    raw_selected = (
        data.get("selected_competitors")
        or data.get("competitors")
        or []
    )

    if not isinstance(raw_selected, list):
        raw_selected = []

    candidate_map = {
        get_root_domain(item.domain): item
        for item in shortlist
    }

    selected = []
    seen = set()

    for item in raw_selected:
        if not isinstance(item, dict):
            continue

        domain = get_root_domain(
            item.get("domain")
            or item.get("official_url")
            or ""
        )

        candidate = candidate_map.get(
            domain
        )

        if (
            candidate is None
            or domain in seen
        ):
            continue

        name = str(
            item.get("brand_name")
            or item.get("name")
            or candidate.domain
        ).strip()

        if not name:
            name = candidate.domain

        official_url = normalize_public_url(
            item.get("official_url")
            or candidate.homepage_url
        )

        selected.append(
            SelectedCompetitor(
                brand_name=name,
                domain=candidate.domain,
                official_url=(
                    official_url
                    or candidate.homepage_url
                ),
                reason=str(
                    item.get("reason")
                    or ""
                ).strip(),
                confidence=(
                    normalize_confidence(
                        item.get(
                            "confidence"
                        )
                    )
                ),
            )
        )

        seen.add(domain)

        if len(selected) == (
            required_count
        ):
            break

    # Graceful fallback to the strongest
    # remaining SERP candidates.
    for candidate in shortlist:
        if len(selected) == (
            required_count
        ):
            break

        if candidate.domain in seen:
            continue

        selected.append(
            SelectedCompetitor(
                brand_name=(
                    candidate.domain
                ),
                domain=candidate.domain,
                official_url=(
                    candidate.homepage_url
                ),
                reason=(
                    "Selected from aggregated "
                    "SERP frequency and rank."
                ),
                confidence=0.5,
            )
        )

        seen.add(candidate.domain)

    return {
        "selected": selected[
            :required_count
        ],
        "rejected": (
            data.get(
                "rejected_candidates",
                [],
            )
            if isinstance(
                data.get(
                    "rejected_candidates",
                    [],
                ),
                list,
            )
            else []
        ),
    }


def select_competitors_stage(
    target_brand,
    candidates,
    keywords,
):
    prompt, shortlist = (
        build_competitor_selection_prompt(
            target_brand=target_brand,
            candidates=candidates,
            keywords=keywords,
        )
    )

    try:
        record = (
            bd_client.google_ai_mode(
                prompt
            )
        )

        parsed = parse_ai_json(
            bd_client.answer_text(record)
        )

        normalized = (
            normalize_selected_competitors(
                data=parsed,
                shortlist=shortlist,
                required_count=5,
            )
        )

        return {
            **normalized,
            "record": record,
            "prompt": prompt,
            "used_fallback": False,
            "shortlist": shortlist,
        }

    except Exception as exc:
        if DEBUG_MODE:
            console.print(
                f"[yellow]Competitor AI "
                f"selection failed: {exc}"
                f"[/yellow]"
            )

        selected = [
            SelectedCompetitor(
                brand_name=item.domain,
                domain=item.domain,
                official_url=(
                    item.homepage_url
                ),
                reason=(
                    "Selected from aggregated "
                    "SERP frequency and rank."
                ),
                confidence=0.5,
            )
            for item in shortlist[:5]
        ]

        return {
            "selected": selected,
            "rejected": [],
            "record": None,
            "prompt": prompt,
            "used_fallback": True,
            "shortlist": shortlist,
            "error": str(exc),
        }


def build_profile_prompt(
    job,
    target_brand,
):
    if job["role"] == "target":
        role_instruction = (
            "This is the audited target. Describe the current "
            "offering and how it is positioned for customers."
        )

        direct_competitor = "false"

    else:
        role_instruction = (
            f"This is being evaluated as an alternative to "
            f"{target_brand.brand_name}. Focus on why a customer "
            f"might compare the two. Selection reason: "
            f"{job['reason']}"
        )

        direct_competitor = "true"

    return f"""
Analyze this current public website.

Name: {job["brand_name"]}
Website: {job["official_url"]}
Domain: {job["domain"]}

{role_instruction}

Adapt the analysis to the actual category. Relevant details may include
products, services, benefits, features, claims, ingredients,
specifications, use cases, audience, price, availability, proof,
reviews, trust signals, location, or delivery model.

Return only JSON:

{{
  "brand_name": "canonical brand, product, service, or organization",
  "official_url": "{job["official_url"]}",
  "domain": "{job["domain"]}",
  "category": "specific category",
  "positioning": "one customer-facing sentence",
  "target_customers": ["customer, user, or audience"],
  "relevant_products": ["relevant product, service, or offering"],
  "key_features": ["feature, benefit, claim, ingredient, or attribute"],
  "differentiators": ["meaningful differentiator"],
  "pricing_model": "price, price tier, pricing approach, or unknown",
  "competitor_reason": "why customers would compare it",
  "direct_competitor": {direct_competitor},
  "confidence": 0.0,
  "evidence": ["specific public evidence"]
}}

Keep lists to five items or fewer. Do not invent claims or pricing.
Return JSON only without Markdown fences.
""".strip()



def normalize_brand_profile(
    data,
    job,
):
    if not isinstance(data, dict):
        data = {}

    name = str(
        data.get("brand_name")
        or data.get("name")
        or job["brand_name"]
    ).strip()

    if not name:
        name = job["brand_name"]

    official_url = normalize_public_url(
        data.get("official_url")
        or data.get("website")
        or job["official_url"]
    )

    domain = get_root_domain(
        data.get("domain")
        or official_url
        or job["domain"]
    )

    normalized = {
        "brand_name": name,
        "official_url": (
            official_url
            or job["official_url"]
        ),
        "domain": (
            domain
            or job["domain"]
        ),
        "category": str(
            data.get("category")
            or ""
        ).strip(),
        "positioning": str(
            data.get("positioning")
            or ""
        ).strip(),
        "target_customers": (
            ensure_string_list(
                data.get(
                    "target_customers"
                )
            )[:5]
        ),
        "relevant_products": (
            ensure_string_list(
                data.get(
                    "relevant_products"
                )
                or data.get("products")
            )[:5]
        ),
        "key_features": (
            ensure_string_list(
                data.get(
                    "key_features"
                )
            )[:5]
        ),
        "differentiators": (
            ensure_string_list(
                data.get(
                    "differentiators"
                )
            )[:5]
        ),
        "pricing_model": str(
            data.get("pricing_model")
            or "unknown"
        ).strip(),
        "competitor_reason": (
            "Target company"
            if job["role"] == "target"
            else str(
                data.get(
                    "competitor_reason"
                )
                or job["reason"]
                or ""
            ).strip()
        ),
        "direct_competitor": (
            job["role"]
            == "competitor"
        ),
        "confidence": (
            normalize_confidence(
                data.get("confidence")
            )
        ),
        "evidence": (
            ensure_string_list(
                data.get("evidence")
            )[:5]
        ),
    }

    return validate_model(
        BrandProfile,
        normalized,
    )


def generate_profile_sync(
    job,
    target_brand,
):
    prompt = build_profile_prompt(
        job,
        target_brand,
    )

    try:
        record = (
            bd_client.google_ai_mode(
                prompt,
                timeout_seconds=600,
            )
        )

        parsed = parse_ai_json(
            bd_client.answer_text(record)
        )

        return {
            "status": "success",
            "job": job,
            "profile": (
                normalize_brand_profile(
                    parsed,
                    job,
                )
            ),
            "record": record,
            "prompt": prompt,
            "error": None,
            "snapshot_id": None,
        }

    except SnapshotTimeoutError as exc:
        return {
            "status": "pending",
            "job": job,
            "profile": None,
            "record": None,
            "prompt": prompt,
            "error": str(exc),
            "snapshot_id": (
                exc.snapshot_id
            ),
        }

    except Exception as exc:
        return {
            "status": "failed",
            "job": job,
            "profile": None,
            "record": None,
            "prompt": prompt,
            "error": str(exc),
            "snapshot_id": None,
        }


def recover_profile_sync(
    task_result,
):
    snapshot_id = task_result[
        "snapshot_id"
    ]

    try:
        records = (
            bd_client.wait_for_snapshot(
                snapshot_id,
                timeout_seconds=600,
            )
        )

        for record in records:
            answer = (
                bd_client.answer_text(
                    record
                )
            )

            if not answer:
                continue

            parsed = parse_ai_json(
                answer
            )

            return {
                **task_result,
                "status": "success",
                "profile": (
                    normalize_brand_profile(
                        parsed,
                        task_result["job"],
                    )
                ),
                "record": record,
                "error": None,
            }

        raise BrightDataAPIError(
            "Recovered snapshot returned "
            "no answer text."
        )

    except Exception as exc:
        return {
            **task_result,
            "status": "failed",
            "error": str(exc),
        }


def fallback_profile(
    job,
    target_brand,
):
    if job["role"] == "target":
        return BrandProfile(
            brand_name=(
                target_brand.brand_name
            ),
            official_url=(
                target_brand.official_url
            ),
            domain=target_brand.domain,
            category=target_brand.category,
            positioning=(
                target_brand.positioning
            ),
            target_customers=(
                target_brand.target_customers
            ),
            relevant_products=(
                target_brand.products
            ),
            key_features=(
                target_brand.key_features
            ),
            differentiators=(
                target_brand.differentiators
            ),
            pricing_model="unknown",
            competitor_reason=(
                "Target company"
            ),
            direct_competitor=False,
            confidence=(
                target_brand.confidence
            ),
            evidence=(
                target_brand.evidence
            ),
        )

    return BrandProfile(
        brand_name=job["brand_name"],
        official_url=job[
            "official_url"
        ],
        domain=job["domain"],
        category="",
        positioning="",
        target_customers=[],
        relevant_products=[],
        key_features=[],
        differentiators=[],
        pricing_model="unknown",
        competitor_reason=(
            job["reason"]
        ),
        direct_competitor=True,
        confidence=0.0,
        evidence=[],
    )


async def run_profile_stage(
    target_brand,
    selected_competitors,
):
    jobs = [
        {
            "role": "target",
            "brand_name": (
                target_brand.brand_name
            ),
            "official_url": (
                target_brand.official_url
            ),
            "domain": (
                target_brand.domain
            ),
            "reason": "",
        }
    ]

    jobs.extend(
        {
            "role": "competitor",
            "brand_name": (
                competitor.brand_name
            ),
            "official_url": (
                competitor.official_url
            ),
            "domain": (
                competitor.domain
            ),
            "reason": (
                competitor.reason
            ),
        }
        for competitor in (
            selected_competitors
        )
    )

    tasks = [
        asyncio.to_thread(
            generate_profile_sync,
            job,
            target_brand,
        )
        for job in jobs
    ]

    results = await asyncio.gather(
        *tasks
    )

    pending = [
        result
        for result in results
        if result["status"]
        == "pending"
    ]

    if pending:
        console.print(
            f"      Waiting for "
            f"{len(pending)} late profile "
            f"snapshot(s)..."
        )

        recovered = await asyncio.gather(
            *[
                asyncio.to_thread(
                    recover_profile_sync,
                    result,
                )
                for result in pending
            ]
        )

        recovered_by_domain = {
            item["job"]["domain"]: item
            for item in recovered
        }

        results = [
            recovered_by_domain.get(
                item["job"]["domain"],
                item,
            )
            if item["status"]
            == "pending"
            else item
            for item in results
        ]

    profiles = []

    for result in results:
        profile = result.get(
            "profile"
        )

        if profile is None:
            profile = fallback_profile(
                result["job"],
                target_brand,
            )

            result["profile"] = profile
            result["used_fallback"] = True

        else:
            result["used_fallback"] = False

        profiles.append(profile)

    # Deduplicate by role/domain.
    deduplicated = []
    seen = set()

    for profile in profiles:
        key = (
            profile.direct_competitor,
            get_root_domain(
                profile.domain
            ),
        )

        if key in seen:
            continue

        seen.add(key)
        deduplicated.append(
            profile
        )

    target_profile = next(
        (
            profile
            for profile in deduplicated
            if not profile.direct_competitor
        ),
        fallback_profile(
            jobs[0],
            target_brand,
        ),
    )

    competitor_profiles = [
        profile
        for profile in deduplicated
        if profile.direct_competitor
    ]

    return {
        "target_profile": (
            target_profile
        ),
        "competitor_profiles": (
            competitor_profiles
        ),
        "all_profiles": [
            target_profile,
            *competitor_profiles,
        ],
        "task_results": results,
        "successful": sum(
            result["status"]
            == "success"
            for result in results
        ),
        "fallbacks": sum(
            result.get(
                "used_fallback",
                False,
            )
            for result in results
        ),
    }


console.print(
    "[bold green]✓ Analysis pipeline loaded[/bold green]"
)


✓ Analysis pipeline loaded

In [8]:
#@title 3C. Load visibility, reporting, and orchestration engine
#@markdown Internal AI visibility, evidence synthesis, checkpoints, report display, and export.
#@markdown This cell normally does not need to be edited.

# ============================================================
# Visibility and reporting
# ============================================================

# ============================================================
# AI visibility prompt
# ============================================================

def build_visibility_prompt(
    target_profile,
    keywords,
):
    category = (
        target_profile.category
        or "product or service category"
    )

    audit_focus = str(
        AUDIT_SETTINGS.get(
            "audit_focus",
            "",
        )
        or ""
    ).strip()

    keyword_text = "; ".join(
        keywords
    )

    prompt = f"""
I am evaluating alternatives in this category:

{category}

Specific need or focus:
{audit_focus or "the needs represented by the searches below"}

Customer searches and requirements:

{keyword_text}

Recommend the leading brands, products, services, providers, or
organizations a real customer should consider.

Compare them using criteria that actually matter in this category.
These may include suitability, benefits, quality, evidence, claims,
ingredients, specifications, features, price, availability,
reputation, service model, performance, ease of use, or other
category-relevant factors.

Ignore criteria that are irrelevant to this market.

Use current public web information. Provide an independent shortlist.
Do not ask follow-up questions.
""".strip()

    if len(prompt) > 4096:
        raise ValueError(
            f"AI visibility prompt is too long: "
            f"{len(prompt)} characters."
        )

    return prompt



# ============================================================
# Brand mention detection
# ============================================================

GENERIC_ALIAS_WORDS = {
    "brand",
    "company",
    "group",
    "global",
    "international",
    "official",
    "online",
    "product",
    "products",
    "service",
    "services",
    "platform",
    "store",
    "shop",
    "market",
    "marketplace",
}



def build_brand_aliases(
    profile,
):
    """
    Build conservative aliases for brands, products, services, and
    organizations across different industries.
    """
    aliases = set()

    brand_name = str(
        profile.brand_name
        or ""
    ).strip()

    brand_lower = (
        brand_name.lower()
    )

    if brand_lower:
        aliases.add(
            brand_lower
        )

        # Normalize punctuation differences such as:
        # La Roche-Posay -> la roche posay
        normalized_brand = re.sub(
            r"[^a-z0-9]+",
            " ",
            brand_lower,
        ).strip()

        if (
            normalized_brand
            and normalized_brand
            != brand_lower
        ):
            aliases.add(
                normalized_brand
            )

    root_domain = get_root_domain(
        profile.domain
    )

    domain_stem = (
        root_domain.split(".")[0]
        if root_domain
        else ""
    )

    normalized_domain_stem = re.sub(
        r"[^a-z0-9]+",
        " ",
        domain_stem.lower(),
    ).strip()

    generic_terms = {
        "amazon",
        "azure",
        "google",
        "microsoft",
        "oracle",
        "company",
        "group",
        "global",
        "international",
        "product",
        "products",
        "service",
        "services",
        "store",
        "shop",
        "official",
        "online",
    }

    if (
        len(normalized_domain_stem) >= 4
        and normalized_domain_stem
        not in generic_terms
    ):
        aliases.add(
            normalized_domain_stem
        )

    words = re.findall(
        r"[A-Za-z0-9]+",
        brand_name,
    )

    normalized_words = [
        word.lower()
        for word in words
    ]

    for index, original_word in enumerate(
        words
    ):
        word = original_word.lower()

        has_internal_capital = any(
            character.isupper()
            for character
            in original_word[1:]
        )

        looks_like_named_product = (
            has_internal_capital
            or (
                len(word) >= 6
                and word.endswith("db")
            )
            or word
            == normalized_domain_stem
        )

        if (
            word not in generic_terms
            and looks_like_named_product
        ):
            aliases.add(word)

        # Preserve distinctive multi-word product names.
        if (
            index + 1
            < len(normalized_words)
            and normalized_words[
                index + 1
            ] == "db"
            and word
            not in generic_terms
        ):
            aliases.add(
                f"{word} db"
            )

    return sorted(
        aliases,
        key=len,
        reverse=True,
    )



def find_brand_mentions(
    answer,
    profiles,
):
    answer = str(
        answer or ""
    )

    results = []

    for profile in profiles:
        aliases = build_brand_aliases(
            profile
        )

        all_positions = []
        matched_aliases = []

        for alias in aliases:
            matches = list(
                re.finditer(
                    rf"\b{re.escape(alias)}\b",
                    answer,
                    flags=re.IGNORECASE,
                )
            )

            if not matches:
                continue

            matched_aliases.append(
                alias
            )

            all_positions.extend(
                match.start()
                for match in matches
            )

        # Multiple aliases can match the same occurrence,
        # so deduplicate character positions.
        unique_positions = sorted(
            set(all_positions)
        )

        results.append(
            {
                "brand_name": (
                    profile.brand_name
                ),
                "domain": profile.domain,
                "role": (
                    "competitor"
                    if profile.direct_competitor
                    else "target"
                ),
                "mentioned": bool(
                    unique_positions
                ),
                "mention_count": len(
                    unique_positions
                ),
                "first_position": (
                    unique_positions[0]
                    if unique_positions
                    else None
                ),
                "matched_aliases": sorted(
                    set(matched_aliases)
                ),
            }
        )

    results.sort(
        key=lambda item: (
            not item["mentioned"],
            (
                item["first_position"]
                if item["first_position"]
                is not None
                else float("inf")
            ),
        )
    )

    return results


def mention_order(
    mentions,
):
    return [
        item["brand_name"]
        for item in mentions
        if item["mentioned"]
    ]


# ============================================================
# Run visibility stage
# ============================================================

async def run_visibility_stage(
    target_profile,
    all_profiles,
    keywords,
):
    prompt = build_visibility_prompt(
        target_profile=target_profile,
        keywords=keywords,
    )

    async def run_engine(
        engine,
    ):
        started_at = time.monotonic()

        try:
            result = await asyncio.to_thread(
                bd_client.race_ai_engine,
                engine,
                prompt,
                3,
                600,
            )

            result[
                "duration_seconds"
            ] = round(
                time.monotonic()
                - started_at,
                2,
            )

            return result

        except Exception as exc:
            return {
                "engine": engine,
                "engine_name": (
                    "ChatGPT"
                    if engine == "chatgpt"
                    else "Gemini"
                ),
                "status": "failed",
                "answer": "",
                "citations": [],
                "web_search_triggered": None,
                "duration_seconds": round(
                    time.monotonic()
                    - started_at,
                    2,
                ),
                "error": str(exc),
            }

    chatgpt_result, gemini_result = (
        await asyncio.gather(
            run_engine("chatgpt"),
            run_engine("gemini"),
        )
    )

    engine_results = {
        "chatgpt": chatgpt_result,
        "gemini": gemini_result,
    }

    mentions = {}

    for engine, result in (
        engine_results.items()
    ):
        if result.get("status") == (
            "success"
        ):
            mentions[engine] = (
                find_brand_mentions(
                    result.get(
                        "answer",
                        "",
                    ),
                    all_profiles,
                )
            )
        else:
            mentions[engine] = []

    return {
        "prompt": prompt,
        "engines": engine_results,
        "mentions": mentions,
    }


# ============================================================
# SERP visibility metrics
# ============================================================

def calculate_serp_metrics(
    domain,
    keyword_serp_results,
):
    target_root = get_root_domain(
        domain
    )

    appearances = []

    for keyword_result in (
        keyword_serp_results
    ):
        if not keyword_result.get(
            "success"
        ):
            continue

        keyword = keyword_result[
            "keyword"
        ]

        for position, result in enumerate(
            keyword_result.get(
                "results",
                [],
            ),
            start=1,
        ):
            result_root = (
                get_root_domain(
                    result.get("domain")
                    or result.get("url")
                    or ""
                )
            )

            if result_root != target_root:
                continue

            try:
                rank = int(
                    result.get(
                        "rank",
                        position,
                    )
                )
            except Exception:
                rank = position

            appearances.append(
                {
                    "keyword": keyword,
                    "rank": rank,
                    "url": result.get(
                        "url",
                        "",
                    ),
                }
            )

            break

    ranks = [
        item["rank"]
        for item in appearances
    ]

    total_keywords = len(
        keyword_serp_results
    )

    return {
        "domain": target_root,
        "appearances": len(
            appearances
        ),
        "total_keywords": (
            total_keywords
        ),
        "coverage": (
            len(appearances)
            / total_keywords
            if total_keywords
            else 0
        ),
        "best_rank": (
            min(ranks)
            if ranks
            else None
        ),
        "average_rank": (
            round(
                sum(ranks) / len(ranks),
                2,
            )
            if ranks
            else None
        ),
        "details": appearances,
    }


def calculate_all_serp_metrics(
    profiles,
    keyword_serp_results,
):
    return {
        profile.domain: (
            calculate_serp_metrics(
                profile.domain,
                keyword_serp_results,
            )
        )
        for profile in profiles
    }


# ============================================================
# Compact, priority-preserving evidence
# ============================================================

def format_serp_metric(
    metric,
):
    appearances = metric[
        "appearances"
    ]

    total = metric[
        "total_keywords"
    ]

    if appearances == 0:
        return f"0/{total} SERPs"

    return (
        f"{appearances}/{total} SERPs, "
        f"best #{metric['best_rank']}, "
        f"avg {metric['average_rank']}"
    )


def format_engine_visibility(
    engine_name,
    result,
    mentions,
):
    if result.get("status") != (
        "success"
    ):
        return (
            f"{engine_name}: failed; "
            f"error={shorten(result.get('error'), 80)}"
        )

    ordered = mention_order(
        mentions
    )

    absent = [
        item["brand_name"]
        for item in mentions
        if not item["mentioned"]
    ]

    target_mention = next(
        (
            item
            for item in mentions
            if item["role"] == "target"
        ),
        None,
    )

    return (
        f"{engine_name}: "
        f"known_order="
        f"{' > '.join(ordered) or 'none'}; "
        f"target_mentions="
        f"{target_mention['mention_count'] if target_mention else 0}; "
        f"absent="
        f"{', '.join(absent) or 'none'}; "
        f"citations="
        f"{len(result.get('citations', []))}; "
        f"web_search="
        f"{result.get('web_search_triggered')}."
    )


def build_report_evidence(
    target_profile,
    competitor_profiles,
    keywords,
    serp_metrics,
    visibility,
):
    """
    Build a compact evidence packet without arbitrary head/tail
    truncation.

    Priority order:
    1. Target identity and positioning
    2. All six brands' SERP metrics
    3. Both AI-engine measurements
    4. All eight keywords
    5. Optional profile descriptions
    """
    lines = []

    lines.append(
        "TARGET:"
        f"{target_profile.brand_name}|"
        f"{target_profile.domain}|"
        f"{shorten(target_profile.category, 60)}|"
        f"{shorten(target_profile.positioning, 150)}"
    )

    target_metric = (
        serp_metrics.get(
            target_profile.domain,
            {},
        )
    )

    lines.append(
        "TARGET_SERP:"
        + format_serp_metric(
            target_metric
        )
    )

    lines.append(
        "TARGET_STRENGTHS:"
        + shorten(
            "; ".join(
                target_profile.differentiators[
                    :4
                ]
            ),
            220,
        )
    )

    lines.append(
        "KEYWORDS:"
        + ";".join(
            shorten(keyword, 48)
            for keyword in keywords
        )
    )

    lines.append(
        "COMPETITORS:"
    )

    for profile in (
        competitor_profiles
    ):
        metric = serp_metrics.get(
            profile.domain,
            {
                "appearances": 0,
                "total_keywords": len(
                    keywords
                ),
                "best_rank": None,
                "average_rank": None,
            },
        )

        lines.append(
            f"-{profile.brand_name}|"
            f"{profile.domain}|"
            f"{format_serp_metric(metric)}|"
            f"{shorten(profile.category, 42)}|"
            f"{shorten(profile.positioning, 75)}"
        )

    lines.append(
        "AI_VISIBILITY:"
    )

    for engine, display_name in (
        ("chatgpt", "ChatGPT"),
        ("gemini", "Gemini"),
    ):
        lines.append(
            format_engine_visibility(
                engine_name=display_name,
                result=visibility[
                    "engines"
                ].get(engine, {}),
                mentions=visibility[
                    "mentions"
                ].get(engine, []),
            )
        )

    evidence = "\n".join(
        lines
    )

    # The fixed structure above should remain compact.
    # If an unusually long company name still pushes it over the
    # budget, shorten only descriptive text from the end of lines.
    if len(evidence) > 1900:
        shortened_lines = []

        for line in lines:
            if line.startswith(
                "TARGET_STRENGTHS:"
            ):
                shortened_lines.append(
                    shorten(line, 140)
                )
            elif line.startswith("-"):
                shortened_lines.append(
                    shorten(line, 145)
                )
            else:
                shortened_lines.append(
                    line
                )

        evidence = "\n".join(
            shortened_lines
        )

    if len(evidence) > 2000:
        raise ValueError(
            f"Structured report evidence is unexpectedly "
            f"long: {len(evidence)} characters."
        )

    return evidence


# ============================================================
# Final report prompt
# ============================================================

def build_final_report_prompt(
    evidence,
):
    prompt = f"""
Using only the evidence below, write a professional Markdown
competitive visibility audit. Adapt the language and recommendations
to the actual product, service, brand, or market category.

Required sections:

# Competitive Visibility Audit
## Executive Summary
## Competitive Landscape
## Google Search Visibility
## AI Answer-Engine Visibility
## Positioning and Information Gaps
## Prioritized Recommendations
## Methodology and Limitations

Requirements:
- Compare the target with all five competitors.
- Use the supplied SERP metrics for every brand.
- Compare measured ChatGPT and Gemini visibility.
- Separate measurements from inference.
- Use evaluation criteria relevant to the actual category.
- Give six practical recommendations.
- Each recommendation must include Priority, Action, Evidence, and
  Expected Impact.
- Do not assume this is a software or enterprise market.
- Mention that SERPs and AI answers vary by time, country, and model.
- Do not invent metrics, claims, ingredients, specifications, prices,
  or causal relationships.
- Do not add a source appendix.

EVIDENCE
--------
{evidence}
--------
END EVIDENCE
""".strip()

    if len(prompt) > 4000:
        raise ValueError(
            f"Final report prompt is too long: "
            f"{len(prompt)} characters."
        )

    return prompt



def generate_report_stage(
    target_profile,
    competitor_profiles,
    keywords,
    keyword_serp_results,
    visibility,
):
    all_profiles = [
        target_profile,
        *competitor_profiles,
    ]

    serp_metrics = (
        calculate_all_serp_metrics(
            profiles=all_profiles,
            keyword_serp_results=(
                keyword_serp_results
            ),
        )
    )

    evidence = build_report_evidence(
        target_profile=target_profile,
        competitor_profiles=(
            competitor_profiles
        ),
        keywords=keywords,
        serp_metrics=serp_metrics,
        visibility=visibility,
    )

    prompt = build_final_report_prompt(
        evidence
    )

    report_result = (
        bd_client.generate_chatgpt_report(
            prompt=prompt,
            timeout_seconds=600,
        )
    )

    return {
        "report": report_result[
            "answer"
        ],
        "record": report_result[
            "record"
        ],
        "snapshot_id": (
            report_result[
                "snapshot_id"
            ]
        ),
        "prompt": prompt,
        "evidence": evidence,
        "serp_metrics": (
            serp_metrics
        ),
    }


# ============================================================
# Source appendix
# ============================================================

def collect_visibility_sources(
    visibility,
    max_per_engine=12,
):
    collected = []
    seen = set()

    for engine, display_name in (
        ("chatgpt", "ChatGPT"),
        ("gemini", "Gemini"),
    ):
        result = visibility[
            "engines"
        ].get(engine, {})

        engine_count = 0

        for citation in result.get(
            "citations",
            [],
        ):
            if not isinstance(
                citation,
                dict,
            ):
                continue

            raw_url = str(
                citation.get("url")
                or citation.get("link")
                or ""
            ).strip()

            canonical_url = (
                canonical_source_url(
                    raw_url
                )
            )

            if (
                not canonical_url
                or canonical_url in seen
            ):
                continue

            seen.add(canonical_url)

            title = str(
                citation.get("title")
                or citation.get("name")
                or citation.get("domain")
                or "Untitled source"
            ).strip()

            collected.append(
                {
                    "engine": (
                        display_name
                    ),
                    "title": title,
                    "url": raw_url,
                    "canonical_url": (
                        canonical_url
                    ),
                }
            )

            engine_count += 1

            if engine_count >= (
                max_per_engine
            ):
                break

    return collected


def build_source_appendix(
    sources,
):
    lines = [
        "",
        "",
        "## Observed AI Sources",
        "",
        (
            "Sources returned by the measured "
            "ChatGPT and Gemini visibility queries."
        ),
        "",
    ]

    if not sources:
        lines.append(
            "No citation records were returned."
        )

        return "\n".join(lines)

    grouped = defaultdict(list)

    for source in sources:
        grouped[
            source["engine"]
        ].append(source)

    for engine in (
        "ChatGPT",
        "Gemini",
    ):
        engine_sources = grouped.get(
            engine,
            [],
        )

        if not engine_sources:
            continue

        lines.append(
            f"### {engine}"
        )
        lines.append("")

        for index, source in enumerate(
            engine_sources,
            start=1,
        ):
            title = (
                source["title"]
                .replace("[", "")
                .replace("]", "")
            )

            lines.append(
                f"{index}. [{title}]"
                f"({source['url']})"
            )

        lines.append("")

    return "\n".join(lines)


def finalize_report(
    report,
    visibility,
):
    sources = (
        collect_visibility_sources(
            visibility
        )
    )

    appendix = (
        build_source_appendix(
            sources
        )
    )

    clean_report = (
        remove_ai_boilerplate(
            report
        )
    )

    if "## Observed AI Sources" in (
        clean_report
    ):
        complete_report = clean_report
    else:
        complete_report = (
            clean_report.rstrip()
            + appendix
        )

    return {
        "report": complete_report,
        "sources": sources,
    }


# ============================================================
# Orchestration and export
# ============================================================

# ============================================================
# File helpers
# ============================================================

def clean_record_for_storage(
    record,
):
    """
    Remove unnecessarily large UI fields before writing records.
    """
    if not isinstance(record, dict):
        return record

    excluded_fields = {
        "answer_html",
        "additional_answer_html",
        "screenshot",
        "html",
    }

    return {
        key: value
        for key, value in record.items()
        if key not in excluded_fields
    }


def write_json(
    path,
    data,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    return path


def write_text(
    path,
    text,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            str(text or "")
        )

    return path


def create_audit_zip(
    output_directory,
):
    output_directory = Path(
        output_directory
    )

    zip_path = (
        output_directory.parent
        / f"{output_directory.name}.zip"
    )

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=(
            zipfile.ZIP_DEFLATED
        ),
    ) as archive:
        for path in (
            output_directory.rglob("*")
        ):
            if not path.is_file():
                continue

            archive.write(
                path,
                arcname=path.relative_to(
                    output_directory
                ),
            )

    return zip_path


# ============================================================
# Progress display
# ============================================================

def print_stage(
    stage_number,
    title,
):
    console.print(
        f"\n[bold cyan]"
        f"[{stage_number}/6] "
        f"{title}"
        f"[/bold cyan]"
    )


def print_stage_success(
    message,
):
    console.print(
        f"      [bold green]✓ "
        f"{message}[/bold green]"
    )


def print_stage_warning(
    message,
):
    console.print(
        f"      [bold yellow]⚠ "
        f"{message}[/bold yellow]"
    )


def format_duration(
    seconds,
):
    if seconds < 60:
        return f"{seconds:.1f}s"

    minutes = int(
        seconds // 60
    )

    remaining = int(
        seconds % 60
    )

    return (
        f"{minutes}m {remaining}s"
    )


# ============================================================
# Visibility serialization
# ============================================================

def serialize_engine_result(
    result,
):
    if not isinstance(result, dict):
        return result

    serialized = {
        key: value
        for key, value in result.items()
        if key != "record"
    }

    if isinstance(
        result.get("record"),
        dict,
    ):
        serialized["record"] = (
            clean_record_for_storage(
                result["record"]
            )
        )

    return serialized


def serialize_profile_task(
    result,
):
    return {
        "status": result.get(
            "status"
        ),
        "job": result.get(
            "job"
        ),
        "profile": (
            model_to_dict(
                result["profile"]
            )
            if result.get("profile")
            is not None
            else None
        ),
        "error": result.get(
            "error"
        ),
        "snapshot_id": result.get(
            "snapshot_id"
        ),
        "used_fallback": result.get(
            "used_fallback",
            False,
        ),
        "record": (
            clean_record_for_storage(
                result["record"]
            )
            if isinstance(
                result.get("record"),
                dict,
            )
            else None
        ),
    }


# ============================================================
# Main pipeline
# ============================================================

async def run_competitive_visibility_audit(
    settings,
):
    """
    Run the complete Competitive Visibility Audit.

    Stages:
    1. Company analysis and keyword generation
    2. Eight parallel Google SERPs
    3. Competitor validation and selection
    4. Six parallel company profiles
    5. ChatGPT and Gemini visibility
    6. Final report and export
    """
    settings = dict(settings)

    audit_started_at = (
        time.monotonic()
    )

    run_timestamp = datetime.now(
        timezone.utc
    )

    run_id = (
        f"{slugify(settings['company_name'])}"
        f"-{run_timestamp.strftime('%Y%m%d-%H%M%S')}"
    )

    output_directory = (
        Path("/content")
        / f"competitive-visibility-{run_id}"
    )

    raw_directory = (
        output_directory / "raw"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    stage_durations = {}
    warnings = []

    # --------------------------------------------------------
    # Stage 1: company analysis and keywords
    # --------------------------------------------------------

    print_stage(
        1,
        "Company analysis and buyer keywords",
    )

    stage_started_at = (
        time.monotonic()
    )

    company_result = await asyncio.to_thread(
        analyze_company_stage,
        settings,
    )

    company_intake = company_result[
        "intake"
    ]

    target_brand = company_intake.brand

    keyword_records = (
        company_intake.buyer_intent_keywords
    )

    keywords = [
        item.keyword
        for item in keyword_records
    ]

    stage_durations[
        "company_analysis"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{target_brand.brand_name} analyzed; "
        f"{len(keywords)} buyer keywords generated"
    )

    write_json(
        output_directory
        / "01_company_analysis.json",
        {
            "created_at": (
                run_timestamp.isoformat()
            ),
            "brand": model_to_dict(
                target_brand
            ),
            "buyer_intent_keywords": [
                model_to_dict(item)
                for item in keyword_records
            ],
            "duration_seconds": round(
                stage_durations[
                    "company_analysis"
                ],
                2,
            ),
        },
    )

    write_json(
        raw_directory
        / "01_company_ai_record.json",
        clean_record_for_storage(
            company_result["record"]
        ),
    )

    if isinstance(
        company_result.get(
            "structuring_record"
        ),
        dict,
    ):
        write_json(
            raw_directory
            / "01_company_structuring_record.json",
            clean_record_for_storage(
                company_result[
                    "structuring_record"
                ]
            ),
        )

    keyword_completion = (
        company_result.get(
            "keyword_completion"
        )
    )

    if (
        isinstance(
            keyword_completion,
            dict,
        )
        and isinstance(
            keyword_completion.get(
                "record"
            ),
            dict,
        )
    ):
        write_json(
            raw_directory
            / "01_keyword_completion_record.json",
            clean_record_for_storage(
                keyword_completion[
                    "record"
                ]
            ),
        )


    # --------------------------------------------------------
    # Stage 2: Google SERPs
    # --------------------------------------------------------

    print_stage(
        2,
        "Google search competitor discovery",
    )

    stage_started_at = (
        time.monotonic()
    )

    serp_result = await run_serp_stage(
        keywords=keywords,
        target_domain=(
            target_brand.domain
        ),
    )

    keyword_serp_results = (
        serp_result[
            "keyword_results"
        ]
    )

    competitor_candidates = (
        serp_result["candidates"]
    )

    stage_durations[
        "serp_discovery"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{serp_result['successful']}/"
        f"{len(keywords)} searches completed "
        f"in "
        f"{format_duration(stage_durations['serp_discovery'])}"
    )

    if serp_result["failed"]:
        warning = (
            f"{serp_result['failed']} "
            f"SERP request(s) failed"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    write_json(
        output_directory
        / "02_serp_results.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "keywords": keywords,
            "successful": (
                serp_result["successful"]
            ),
            "failed": (
                serp_result["failed"]
            ),
            "keyword_results": (
                keyword_serp_results
            ),
            "competitor_candidates": [
                model_to_dict(item)
                for item in (
                    competitor_candidates
                )
            ],
            "duration_seconds": round(
                stage_durations[
                    "serp_discovery"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 3: competitor selection
    # --------------------------------------------------------

    print_stage(
        3,
        "Direct competitor selection",
    )

    stage_started_at = (
        time.monotonic()
    )

    selection_result = (
        await asyncio.to_thread(
            select_competitors_stage,
            target_brand,
            competitor_candidates,
            keywords,
        )
    )

    selected_competitors = (
        selection_result["selected"]
    )

    stage_durations[
        "competitor_selection"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    if selection_result.get(
        "used_fallback"
    ):
        warning = (
            "AI competitor validation failed; "
            "SERP-ranked fallback was used"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    for competitor in (
        selected_competitors
    ):
        console.print(
            f"      ✓ "
            f"{competitor.brand_name}"
        )

    write_json(
        output_directory
        / "03_competitor_selection.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "selected_competitors": [
                model_to_dict(item)
                for item in (
                    selected_competitors
                )
            ],
            "rejected_candidates": (
                selection_result.get(
                    "rejected",
                    [],
                )
            ),
            "used_fallback": (
                selection_result.get(
                    "used_fallback",
                    False,
                )
            ),
            "duration_seconds": round(
                stage_durations[
                    "competitor_selection"
                ],
                2,
            ),
        },
    )

    if isinstance(
        selection_result.get("record"),
        dict,
    ):
        write_json(
            raw_directory
            / "03_selection_ai_record.json",
            clean_record_for_storage(
                selection_result["record"]
            ),
        )

    # --------------------------------------------------------
    # Stage 4: profiles
    # --------------------------------------------------------

    print_stage(
        4,
        "Target and competitor profiles",
    )

    stage_started_at = (
        time.monotonic()
    )

    profile_result = (
        await run_profile_stage(
            target_brand=target_brand,
            selected_competitors=(
                selected_competitors
            ),
        )
    )

    target_profile = (
        profile_result[
            "target_profile"
        ]
    )

    competitor_profiles = (
        profile_result[
            "competitor_profiles"
        ]
    )

    all_profiles = (
        profile_result[
            "all_profiles"
        ]
    )

    stage_durations[
        "brand_profiles"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    print_stage_success(
        f"{len(all_profiles)}/6 profiles available"
    )

    if profile_result[
        "fallbacks"
    ]:
        warning = (
            f"{profile_result['fallbacks']} "
            f"profile fallback(s) used"
        )

        warnings.append(warning)
        print_stage_warning(warning)

    write_json(
        output_directory
        / "04_brand_profiles.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "target_profile": (
                model_to_dict(
                    target_profile
                )
            ),
            "competitor_profiles": [
                model_to_dict(item)
                for item in (
                    competitor_profiles
                )
            ],
            "successful_profiles": (
                profile_result[
                    "successful"
                ]
            ),
            "fallback_profiles": (
                profile_result[
                    "fallbacks"
                ]
            ),
            "tasks": [
                serialize_profile_task(
                    result
                )
                for result in (
                    profile_result[
                        "task_results"
                    ]
                )
            ],
            "duration_seconds": round(
                stage_durations[
                    "brand_profiles"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 5: AI visibility
    # --------------------------------------------------------

    print_stage(
        5,
        "ChatGPT and Gemini visibility",
    )

    stage_started_at = (
        time.monotonic()
    )

    visibility_result = (
        await run_visibility_stage(
            target_profile=(
                target_profile
            ),
            all_profiles=all_profiles,
            keywords=keywords,
        )
    )

    stage_durations[
        "ai_visibility"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    for engine in (
        "chatgpt",
        "gemini",
    ):
        engine_result = (
            visibility_result[
                "engines"
            ][engine]
        )

        engine_name = (
            engine_result.get(
                "engine_name",
                engine.title(),
            )
        )

        if engine_result.get(
            "status"
        ) == "success":
            print_stage_success(
                f"{engine_name} completed in "
                f"{format_duration(engine_result['duration_seconds'])}"
            )
        else:
            warning = (
                f"{engine_name} visibility failed: "
                f"{engine_result.get('error', 'unknown error')}"
            )

            warnings.append(warning)
            print_stage_warning(warning)

    write_json(
        output_directory
        / "05_ai_visibility.json",
        {
            "created_at": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "prompt": (
                visibility_result["prompt"]
            ),
            "engines": {
                engine: (
                    serialize_engine_result(
                        result
                    )
                )
                for engine, result in (
                    visibility_result[
                        "engines"
                    ].items()
                )
            },
            "mentions": (
                visibility_result[
                    "mentions"
                ]
            ),
            "duration_seconds": round(
                stage_durations[
                    "ai_visibility"
                ],
                2,
            ),
        },
    )

    # --------------------------------------------------------
    # Stage 6: final report
    # --------------------------------------------------------

    print_stage(
        6,
        "Final report and export",
    )

    stage_started_at = (
        time.monotonic()
    )

    report_result = (
        await asyncio.to_thread(
            generate_report_stage,
            target_profile,
            competitor_profiles,
            keywords,
            keyword_serp_results,
            visibility_result,
        )
    )

    finalized_report = (
        finalize_report(
            report=report_result[
                "report"
            ],
            visibility=(
                visibility_result
            ),
        )
    )

    final_report = (
        finalized_report["report"]
    )

    final_sources = (
        finalized_report["sources"]
    )

    stage_durations[
        "final_report"
    ] = (
        time.monotonic()
        - stage_started_at
    )

    report_markdown_path = (
        output_directory
        / "06_competitive_visibility_audit.md"
    )

    write_text(
        report_markdown_path,
        final_report,
    )

    write_json(
        raw_directory
        / "06_final_report_record.json",
        clean_record_for_storage(
            report_result["record"]
        ),
    )

    print_stage_success(
        f"Report generated in "
        f"{format_duration(stage_durations['final_report'])}"
    )

    # --------------------------------------------------------
    # Final structured audit object
    # --------------------------------------------------------

    completed_at = datetime.now(
        timezone.utc
    )

    total_duration = (
        time.monotonic()
        - audit_started_at
    )

    audit_data = {
        "run_id": run_id,
        "created_at": (
            run_timestamp.isoformat()
        ),
        "completed_at": (
            completed_at.isoformat()
        ),
        "configuration": {
            "company_name": settings[
                "company_name"
            ],
            "company_url": settings[
                "company_url"
            ],
            "company_domain": settings[
                "company_domain"
            ],
            "country": settings[
                "country"
            ],
            "serp_zone": settings[
                "serp_zone"
            ],
            "debug": settings.get(
                "debug",
                False,
            ),
        },
        "target": model_to_dict(
            target_profile
        ),
        "buyer_intent_keywords": [
            model_to_dict(item)
            for item in keyword_records
        ],
        "serp": {
            "keyword_results": (
                keyword_serp_results
            ),
            "metrics": (
                report_result[
                    "serp_metrics"
                ]
            ),
            "competitor_candidates": [
                model_to_dict(item)
                for item in (
                    competitor_candidates
                )
            ],
        },
        "competitor_selection": {
            "selected": [
                model_to_dict(item)
                for item in (
                    selected_competitors
                )
            ],
            "rejected": (
                selection_result.get(
                    "rejected",
                    [],
                )
            ),
            "used_fallback": (
                selection_result.get(
                    "used_fallback",
                    False,
                )
            ),
        },
        "profiles": {
            "target": model_to_dict(
                target_profile
            ),
            "competitors": [
                model_to_dict(item)
                for item in (
                    competitor_profiles
                )
            ],
        },
        "ai_visibility": {
            "prompt": (
                visibility_result["prompt"]
            ),
            "engines": {
                engine: (
                    serialize_engine_result(
                        result
                    )
                )
                for engine, result in (
                    visibility_result[
                        "engines"
                    ].items()
                )
            },
            "mentions": (
                visibility_result[
                    "mentions"
                ]
            ),
        },
        "final_report": {
            "generator": "ChatGPT",
            "web_search": False,
            "snapshot_id": (
                report_result[
                    "snapshot_id"
                ]
            ),
            "evidence": (
                report_result[
                    "evidence"
                ]
            ),
            "prompt": (
                report_result[
                    "prompt"
                ]
            ),
            "markdown": final_report,
            "sources": final_sources,
        },
        "warnings": warnings,
        "durations": {
            **{
                stage: round(
                    duration,
                    2,
                )
                for stage, duration in (
                    stage_durations.items()
                )
            },
            "total_seconds": round(
                total_duration,
                2,
            ),
        },
        "files": {},
    }

    audit_json_path = (
        output_directory
        / "06_competitive_visibility_audit.json"
    )

    audit_data["files"] = {
        "output_directory": str(
            output_directory
        ),
        "markdown_report": str(
            report_markdown_path
        ),
        "json_report": str(
            audit_json_path
        ),
    }

    write_json(
        audit_json_path,
        audit_data,
    )

    zip_path = create_audit_zip(
        output_directory
    )

    audit_data["files"][
        "zip_archive"
    ] = str(zip_path)

    # Rewrite JSON so it includes the ZIP path.
    write_json(
        audit_json_path,
        audit_data,
    )

    print_stage_success(
        "Markdown, JSON and ZIP saved"
    )

    console.print(
        f"\n[bold green]"
        f"✓ Audit complete in "
        f"{format_duration(total_duration)}"
        f"[/bold green]"
    )

    return audit_data


# ============================================================
# Report display
# ============================================================

def display_audit(
    audit,
):
    target = audit["target"]

    console.print(
        "\n[bold cyan]"
        "Competitive Visibility Audit"
        "[/bold cyan]"
    )

    console.print(
        f"Company: "
        f"[bold]{target['brand_name']}[/bold]"
    )
    console.print(
        f"Website: "
        f"{target['official_url']}"
    )
    console.print(
        f"Category: "
        f"{target['category']}"
    )
    console.print(
        f"Country: "
        f"{audit['configuration']['country']}"
    )

    # --------------------------------------------------------
    # Keywords
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Buyer-intent keywords[/bold cyan]"
    )

    keyword_rows = []

    for index, item in enumerate(
        audit[
            "buyer_intent_keywords"
        ],
        start=1,
    ):
        keyword_rows.append(
            {
                "#": index,
                "keyword": item[
                    "keyword"
                ],
                "intent": item[
                    "intent"
                ],
            }
        )

    display(
        pd.DataFrame(
            keyword_rows
        )
    )

    # --------------------------------------------------------
    # Competitors and SERP metrics
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Competitive search visibility[/bold cyan]"
    )

    metrics = audit[
        "serp"
    ]["metrics"]

    profile_rows = []

    all_profile_data = [
        audit["profiles"]["target"],
        *audit["profiles"][
            "competitors"
        ],
    ]

    for profile in all_profile_data:
        metric = metrics.get(
            profile["domain"],
            {},
        )

        profile_rows.append(
            {
                "role": (
                    "competitor"
                    if profile[
                        "direct_competitor"
                    ]
                    else "target"
                ),
                "brand": profile[
                    "brand_name"
                ],
                "domain": profile[
                    "domain"
                ],
                "SERP coverage": (
                    f"{metric.get('appearances', 0)}/"
                    f"{metric.get('total_keywords', 0)}"
                ),
                "best rank": metric.get(
                    "best_rank"
                ),
                "average rank": (
                    metric.get(
                        "average_rank"
                    )
                ),
            }
        )

    display(
        pd.DataFrame(
            profile_rows
        )
    )

    # --------------------------------------------------------
    # AI visibility
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]AI visibility[/bold cyan]"
    )

    visibility_rows = []

    for engine, mentions in (
        audit[
            "ai_visibility"
        ]["mentions"].items()
    ):
        for mention in mentions:
            visibility_rows.append(
                {
                    "engine": (
                        engine.title()
                    ),
                    "role": mention[
                        "role"
                    ],
                    "brand": mention[
                        "brand_name"
                    ],
                    "mentioned": mention[
                        "mentioned"
                    ],
                    "mentions": mention[
                        "mention_count"
                    ],
                    "first position": (
                        mention[
                            "first_position"
                        ]
                    ),
                }
            )

    display(
        pd.DataFrame(
            visibility_rows
        )
    )

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    console.print(
        "\n[bold cyan]Final report[/bold cyan]\n"
    )

    console.print(
        Markdown(
            audit[
                "final_report"
            ]["markdown"]
        )
    )

    # --------------------------------------------------------
    # Files
    # --------------------------------------------------------

    console.print(
        "\n[bold green]Saved files[/bold green]"
    )

    console.print(
        f"Markdown: "
        f"{audit['files']['markdown_report']}"
    )
    console.print(
        f"JSON: "
        f"{audit['files']['json_report']}"
    )
    console.print(
        f"ZIP: "
        f"{audit['files']['zip_archive']}"
    )

    if audit.get("warnings"):
        console.print(
            "\n[bold yellow]Warnings[/bold yellow]"
        )

        for warning in audit[
            "warnings"
        ]:
            console.print(
                f"• {warning}"
            )


def download_audit(
    audit,
):
    from google.colab import files

    zip_path = audit.get(
        "files",
        {},
    ).get(
        "zip_archive"
    )

    if not zip_path:
        raise ValueError(
            "The audit ZIP path is missing."
        )

    if not Path(zip_path).exists():
        raise FileNotFoundError(
            f"Audit ZIP does not exist: "
            f"{zip_path}"
        )

    files.download(zip_path)


console.print(
    "[bold green]✓ Visibility, reporting, and orchestration engine loaded[/bold green]"
)


✓ Visibility, reporting, and orchestration engine loaded

In [9]:
#@title 4. Run Competitive Visibility Audit
#@markdown Run this cell to execute the complete live audit.

# Synchronize the current form settings with the API client.
DEBUG_MODE = bool(
    AUDIT_SETTINGS.get(
        "debug",
        False,
    )
)

bd_client.debug = DEBUG_MODE
bd_client.country = (
    AUDIT_SETTINGS["country"].upper()
)
bd_client.serp_zone = (
    AUDIT_SETTINGS["serp_zone"]
)

console.print(
    f"[bold cyan]Debug logging: "
    f"{'enabled' if DEBUG_MODE else 'disabled'}"
    f"[/bold cyan]"
)

AUDIT_RESULT = await run_competitive_visibility_audit(
    AUDIT_SETTINGS
)

display_audit(
    AUDIT_RESULT
)

if AUDIT_SETTINGS.get(
    "auto_download",
    False,
):
    download_audit(
        AUDIT_RESULT
    )


Debug logging: enabled

[1/6] Company analysis and buyer keywords

Starting Google AI Mode company research

Google AI Mode research returned 9,538 characters

Starting ChatGPT structuring attempt 1/2

...

JSON: /content/competitive-visibility-bright-data-20260909-073624/06_competitive_visibility_audit.json

ZIP: /content/competitive-visibility-bright-data-20260909-073624.zip

Warnings

• 1 SERP request(s) failed

• 1 profile fallback(s) used